# Combined DAM + IFN-R Boxplot

In [1]:
import os
import numpy as np
import scanpy as sc
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Patch
from scipy.stats import ttest_ind


In [10]:
ad_full = sc.read_h5ad("/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/combined/"
                            "sub_adata_microglia_leiden6_noOutside_reclustered_gatex2_with_miranda_DAMvHOME_damifnhome_combined_withCounts_combinedcleaned.h5ad")

In [8]:
# ============================================================
# BOXPLOTS + MOUSE-MEAN CSVs + WELCH P-TESTS
# USING ad_full DIRECTLY
# - Uses ad_full with final annotations already present
# - Uses miranda_cell_type_refined_combined (HOME/DAM)
# - Removes WT from DAM state completely in plots + exported plotting/stats tables
# - Exports mouse means, group means, and Welch p-test results
# ============================================================


# PATHS / SETTINGS
OUT_DIR = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/"
    "geneCounts_mouseMeans_boxplots_from_ad_full"
)
os.makedirs(OUT_DIR, exist_ok=True)

OUT_PDF       = os.path.join(OUT_DIR, "ALLgenes_countsPerCell_mouseMeans.pdf")
OUT_MOUSE_CSV = os.path.join(OUT_DIR, "ALLgenes_mouseMeans_long.csv")
OUT_GROUP_CSV = os.path.join(OUT_DIR, "ALLgenes_groupMeans_long.csv")
OUT_PVAL_CSV  = os.path.join(OUT_DIR, "ALLgenes_mouseMeans_WelchPvals_long.csv")

# columns already in ad_full
STATE_COL   = "miranda_cell_type_refined_combined"   # final HOME/DAM annotation column
COND_COL    = "condition"
COND_CONSOL = "condition_consol"
RUN_COL     = "run_key"
LEIDEN_COL  = "leiden"
COUNTS_LAYER = "counts"

# optional restriction
RESTRICT_LEIDEN = True
LEIDEN_TARGET = "6"

# new/use columns
REPL_COL = "condition_rep"

# plotting orders
cond_order  = ["FAD-Abe", "FAD-Veh", "WT-Abe", "WT-Veh"]
state_order = ["DAM", "HOME"]
keep_states = set(state_order)

# remove WT from DAM entirely
wt_conds = {"WT-Veh", "WT-Abe"}

# comparisons for Welch tests on mouse means
# these will be skipped automatically if a state/condition combo is absent
COMPARISONS = [
    ("FAD-Veh", "WT-Veh"),
    ("FAD-Abe", "FAD-Veh"),
    ("WT-Abe", "WT-Veh"),
]

# colors
cond_palette = {
    "FAD-Abe": "#1f77b4",
    "FAD-Veh": "#ff7f0e",
    "WT-Abe":  "#2ca02c",
    "WT-Veh":  "#d62728",
}

# START FROM ad_full
adata = ad_full.copy()

if RESTRICT_LEIDEN:
    adata = adata[adata.obs[LEIDEN_COL].astype(str) == str(LEIDEN_TARGET)].copy()

if COUNTS_LAYER not in adata.layers:
    raise KeyError(
        f"adata.layers['{COUNTS_LAYER}'] not found in ad_full. "
        f"Available layers: {list(adata.layers.keys())}"
    )

assert STATE_COL in adata.obs, f"Missing {STATE_COL}"
assert COND_COL in adata.obs, f"Missing {COND_COL}"
assert COND_CONSOL in adata.obs, f"Missing {COND_CONSOL}"

# keep only HOME/DAM
adata = adata[adata.obs[STATE_COL].astype(str).isin(keep_states)].copy()
adata.obs[STATE_COL] = (
    adata.obs[STATE_COL].astype(str).astype("category").cat.remove_unused_categories()
)

print("adata shape after filtering:", adata.shape)
print("State counts:")
print(adata.obs[STATE_COL].value_counts())
print("Condition counts:")
print(adata.obs[COND_CONSOL].value_counts())

#  REPLICATE LABEL
cond = adata.obs[COND_COL].astype(str)
adata.obs[REPL_COL] = cond.str.replace(r"^WT-V(\d+)$", r"WT-Veh\1", regex=True)

print("\nExample condition -> condition_rep:")
print(pd.DataFrame({
    "condition": cond.head(8).values,
    "condition_rep": adata.obs[REPL_COL].head(8).values
}))

# BASE OBS TABLE
base_obs = adata.obs[[COND_CONSOL, COND_COL, REPL_COL, STATE_COL]].copy()

# remove WT from DAM entirely so plots/CSVs/stats are all aligned
drop_mask = (
    (base_obs[STATE_COL].astype(str) == "DAM") &
    (base_obs[COND_CONSOL].astype(str).isin(wt_conds))
)

base_obs = base_obs.loc[~drop_mask].copy()

print("\nCounts after removing WT from DAM:")
print(base_obs.groupby([STATE_COL, COND_CONSOL]).size())

# stable condition order among those still present
present_conds = [c for c in cond_order if c in set(base_obs[COND_CONSOL].astype(str))]
extras = sorted(set(base_obs[COND_CONSOL].astype(str)) - set(present_conds))
present_conds = present_conds + extras

print("\npresent_conds:", present_conds)

box_palette_all   = {c: cond_palette.get(c, "#808080") for c in present_conds}
black_palette_all = {c: "black" for c in present_conds}

# GENE COUNTS HELPER
def get_gene_counts(adata_obj, gene, layer=COUNTS_LAYER):
    X = adata_obj[:, gene].layers[layer]
    if sp.issparse(X):
        return np.asarray(X.toarray()).ravel()
    return np.asarray(X).ravel()

# LOOP GENES
genes = adata.var_names.to_list()
print("\nn_genes:", len(genes))
print("Writing PDF:", OUT_PDF)

all_mouse_means = []
all_group_means = []
all_pvals = []

with PdfPages(OUT_PDF) as pdf:
    for i, gene in enumerate(genes, start=1):
        expr_full = get_gene_counts(adata, gene, layer=COUNTS_LAYER)

        # row-aligned expression table
        df = base_obs.copy()
        idx = adata.obs_names.get_indexer(df.index)
        df["expr"] = expr_full[idx]
        df["gene"] = gene

        # mouse mean per (condition_consol × condition_rep × state)
        df_mouse = (
            df.groupby([COND_CONSOL, REPL_COL, STATE_COL], observed=True)["expr"]
              .agg(mouse_mean="mean", n_cells="size")
              .reset_index()
        )
        df_mouse["gene"] = gene
        all_mouse_means.append(df_mouse)

        # group mean across mice
        df_group = (
            df_mouse.groupby([COND_CONSOL, STATE_COL], observed=True)["mouse_mean"]
                   .agg(mean_of_mice="mean", n_mice="count")
                   .reset_index()
        )
        df_group["gene"] = gene
        all_group_means.append(df_group)

        # Welch p-tests on mouse means
        # IMPORTANT: because WT was already removed from DAM in base_obs,
        # DAM tests involving WT conditions will naturally be skipped
        for state in state_order:
            sub = df_mouse[df_mouse[STATE_COL].astype(str) == state].copy()
            if sub.empty:
                continue

            for cond_a, cond_b in COMPARISONS:
                a = sub.loc[sub[COND_CONSOL].astype(str) == cond_a, "mouse_mean"].dropna().to_numpy()
                b = sub.loc[sub[COND_CONSOL].astype(str) == cond_b, "mouse_mean"].dropna().to_numpy()

                if len(a) == 0 or len(b) == 0:
                    continue

                try:
                    stat, pval = ttest_ind(a, b, equal_var=False)
                except Exception:
                    stat, pval = (np.nan, np.nan)

                all_pvals.append({
                    "gene": gene,
                    "state": state,
                    "cond_a": cond_a,
                    "cond_b": cond_b,
                    "comparison": f"{cond_a}_vs_{cond_b}",
                    "mean_mouseMean_a": float(np.mean(a)),
                    "mean_mouseMean_b": float(np.mean(b)),
                    "log2FC_mouseMean": float(np.log2((np.mean(a) + 1e-6) / (np.mean(b) + 1e-6))),
                    "welch_t": stat,
                    "pval": pval,
                    "n_mice_a": int(len(a)),
                    "n_mice_b": int(len(b)),
                })

        # PLOT
        plt.figure(figsize=(8.2, 5.2))

        ax = sns.boxplot(
            data=df_mouse,
            x=STATE_COL,
            y="mouse_mean",
            hue=COND_CONSOL,
            order=state_order,
            hue_order=present_conds,
            fliersize=0,
            palette=box_palette_all
        )

        # mouse mean dots in black
        sns.stripplot(
            data=df_mouse,
            x=STATE_COL,
            y="mouse_mean",
            hue=COND_CONSOL,
            order=state_order,
            hue_order=present_conds,
            dodge=True,
            jitter=0.12,
            size=6,
            alpha=0.95,
            linewidth=0,
            palette=black_palette_all,
            ax=ax
        )

        # mean of mice as diamonds ONLY (no connecting line)
        sns.stripplot(
            data=df_group,
            x=STATE_COL,
            y="mean_of_mice",
            hue=COND_CONSOL,
            order=state_order,
            hue_order=present_conds,
            dodge=True,
            marker="D",
            size=8,
            edgecolor="black",
            linewidth=1,
            palette=box_palette_all,
            ax=ax
        )

        ymax = max(
            float(np.nanmax(df_mouse["mouse_mean"].values)) if len(df_mouse) else 0.0,
            float(np.nanmax(df_group["mean_of_mice"].values)) if len(df_group) else 0.0
        )
        if not np.isfinite(ymax) or ymax <= 0:
            ymax = 0.25
        ax.set_ylim(0, ymax * 1.1)

        ax.set_title(
            f"{gene} (raw counts per cell)\n"
            f"box=mouse means; black dots=mouse means; diamonds=mean across mice"
        )
        ax.set_xlabel("")
        ax.set_ylabel("Mean transcripts per cell (per mouse; raw counts)")

        # clean legend outside plot
        legend_handles = [
            Patch(facecolor=box_palette_all[c], edgecolor="black", label=c)
            for c in present_conds
        ]
        ax.legend(
            handles=legend_handles,
            title="Condition",
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            frameon=False
        )

        plt.tight_layout()
        pdf.savefig(ax.figure, dpi=200)
        plt.close()

        if i % 50 == 0:
            print(f"...{i}/{len(genes)} pages written")

print("Finished PDF:", OUT_PDF)

# EXPORT CSVs
mouse_means_long = pd.concat(all_mouse_means, ignore_index=True)
group_means_long = pd.concat(all_group_means, ignore_index=True)
pvals_long = pd.DataFrame(all_pvals)

mouse_means_long.to_csv(OUT_MOUSE_CSV, index=False)
group_means_long.to_csv(OUT_GROUP_CSV, index=False)
pvals_long.to_csv(OUT_PVAL_CSV, index=False)

print("Wrote mouse means:", OUT_MOUSE_CSV)
print("Wrote group means:", OUT_GROUP_CSV)
print("Wrote Welch p-values:", OUT_PVAL_CSV)
print("Done.")

adata shape after filtering: (48534, 347)
State counts:
miranda_cell_type_refined_combined
HOME    34101
DAM     14433
Name: count, dtype: int64
Condition counts:
condition_consol
FAD-Veh    18151
FAD-Abe    15479
WT-Veh      8135
WT-Abe      6769
Name: count, dtype: int64

Example condition -> condition_rep:
  condition condition_rep
0  FAD-Abe3      FAD-Abe3
1  FAD-Abe3      FAD-Abe3
2  FAD-Abe3      FAD-Abe3
3  FAD-Abe3      FAD-Abe3
4  FAD-Abe3      FAD-Abe3
5  FAD-Abe3      FAD-Abe3
6  FAD-Abe3      FAD-Abe3
7  FAD-Abe3      FAD-Abe3

Counts after removing WT from DAM:
miranda_cell_type_refined_combined  condition_consol
DAM                                 WT-Veh                  0
                                    WT-Abe                  0
                                    FAD-Veh              7447
                                    FAD-Abe              6634
HOME                                WT-Veh               7927
                                    WT-Abe              

/tmp/ipykernel_3714639/262557996.py:114: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(base_obs.groupby([STATE_COL, COND_CONSOL]).size())


...50/347 pages written
...100/347 pages written
...150/347 pages written
...200/347 pages written
...250/347 pages written
...300/347 pages written
Finished PDF: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/geneCounts_mouseMeans_boxplots_from_ad_full/ALLgenes_countsPerCell_mouseMeans.pdf
Wrote mouse means: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/geneCounts_mouseMeans_boxplots_from_ad_full/ALLgenes_mouseMeans_long.csv
Wrote group means: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/geneCounts_mouseMeans_boxplots_from_ad_full/ALLgenes_groupMeans_long.csv
Wrote Welch p-values: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/geneCounts_mouseMeans_boxplots_from_ad_full/ALLgenes_mouseMeans_WelchPvals_long.csv
Done.


In [11]:
ad_full

AnnData object with n_obs × n_vars = 943639 × 347
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level', 'sample_id', 'run_key', 'batch', 'unique_cell_index', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'leiden', 'miranda_cell_type', 'condition', 'condition_consol', 'miranda_cell_type_refined', 'DAM_IFN_HOME_full', 'DAM_HOME_combined_full', 'miranda_cell_type_refined_combined'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'hvg', 'immune_markers_presen

# Violin

In [17]:
# ============================================================
# VIOLIN PLOTS + PER-CELL CSVs + WELCH P-TESTS ON ALL CELLS
# USING FINAL OBJECT
#
# - Uses final object with palette2
# - Uses miranda_cell_type_refined_combinedUpdate (DAM / Homeostatic microglia)
# - Uses ALL CELLS and raw counts from adata.layers["counts"]
# - Removes WT from DAM entirely in plots + exported tables
# - Exports per-cell long table, group summaries, and Welch p-values
# - Adds percent expressing (% cells with expr > 0)
# - Plots ALL black dots (not subsampled)
#
# WRITES THREE PDFs:
#   1) RAW counts per cell
#   2) log1p(counts per cell)
#   3) log1p(counts per cell), clipped at the gene-wise 95th percentile
# ============================================================

import os
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Patch
from scipy.stats import ttest_ind

# ============================================================
# PATHS / SETTINGS
# ============================================================
ADATA_PATH = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/combined/"
    "sub_adata_microglia_leiden6_noOutside_reclustered_gatex2_with_miranda_"
    "DAMvHOME_damifnhome_combined_withCounts_combinedcleaned_UPDATE_palette2.h5ad"
)

OUT_DIR = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/"
    "geneCounts_allCells_violinplots_from_ad_full_palette2_pctExpr_ALLDOTS"
)
os.makedirs(OUT_DIR, exist_ok=True)

OUT_PDF_RAW = os.path.join(
    OUT_DIR,
    "ALLgenes_countsPerCell_allCells_violin_RAW_pctExpr_ALLDOTS.pdf"
)
OUT_PDF_LOG1P = os.path.join(
    OUT_DIR,
    "ALLgenes_countsPerCell_allCells_violin_LOG1P_pctExpr_ALLDOTS.pdf"
)
OUT_PDF_LOG1P_CLIP95 = os.path.join(
    OUT_DIR,
    "ALLgenes_countsPerCell_allCells_violin_LOG1P_CLIP95_pctExpr_ALLDOTS.pdf"
)

OUT_CELL_CSV = os.path.join(
    OUT_DIR,
    "ALLgenes_allCells_long_pctExpr.csv"
)
OUT_GROUP_CSV = os.path.join(
    OUT_DIR,
    "ALLgenes_allCells_groupSummary_long_pctExpr.csv"
)
OUT_PVAL_CSV = os.path.join(
    OUT_DIR,
    "ALLgenes_allCells_WelchPvals_long_pctExpr.csv"
)

STATE_COL = "miranda_cell_type_refined_combinedUpdate"
COND_COL = "condition"
COND_CONSOL = "condition_consol"
RUN_COL = "run_key"
LEIDEN_COL = "leiden"
COUNTS_LAYER = "counts"

RESTRICT_LEIDEN = False
LEIDEN_TARGET = "6"

REPL_COL = "condition_rep"

cond_order = ["FAD-Abe", "FAD-Veh", "WT-Abe", "WT-Veh"]
state_order = ["DAM", "Homeostatic microglia"]
keep_states = set(state_order)

# keep prior logic unless you want to remove it
wt_conds = {"WT-Veh", "WT-Abe"}

COMPARISONS = [
    ("FAD-Veh", "WT-Veh"),
    ("FAD-Abe", "FAD-Veh"),
    ("WT-Abe", "WT-Veh"),
]

cond_palette = {
    "FAD-Abe": "#1f77b4",
    "FAD-Veh": "#ff7f0e",
    "WT-Abe":  "#2ca02c",
    "WT-Veh":  "#d62728",
}

PSEUDOCOUNT = 1e-6

# violin settings
VIOLIN_CUT = 0
VIOLIN_SCALE = "width"
POINT_SIZE = 0.55
POINT_ALPHA = 0.20
POINT_JITTER = 0.18

# clipping settings
CLIP_PERCENTILE = 95
MIN_CLIP_VALUE = None  # leave None unless you want to force a minimum clip cap

# annotation settings
PCT_EXPR_TEXT_SIZE = 8
PCT_EXPR_OFFSET_FRAC = 0.025

# ============================================================
# LOAD OBJECT
# ============================================================
adata = sc.read_h5ad(ADATA_PATH)

if RESTRICT_LEIDEN:
    adata = adata[adata.obs[LEIDEN_COL].astype(str) == str(LEIDEN_TARGET)].copy()

if COUNTS_LAYER not in adata.layers:
    raise KeyError(
        f"adata.layers['{COUNTS_LAYER}'] not found. "
        f"Available layers: {list(adata.layers.keys())}"
    )

for col in [STATE_COL, COND_COL, COND_CONSOL]:
    if col not in adata.obs.columns:
        raise KeyError(f"Missing required obs column: {col}")

# keep only DAM / Homeostatic microglia
adata = adata[adata.obs[STATE_COL].astype(str).isin(keep_states)].copy()
adata.obs[STATE_COL] = (
    adata.obs[STATE_COL]
    .astype(str)
    .astype("category")
    .cat.remove_unused_categories()
)

print("adata shape after filtering:", adata.shape)
print("State counts:")
print(adata.obs[STATE_COL].value_counts())
print("Condition counts:")
print(adata.obs[COND_CONSOL].value_counts())

# ============================================================
# REPLICATE LABEL
# ============================================================
cond = adata.obs[COND_COL].astype(str)
adata.obs[REPL_COL] = cond.str.replace(r"^WT-V(\d+)$", r"WT-Veh\1", regex=True)

print("\nExample condition -> condition_rep:")
print(pd.DataFrame({
    "condition": cond.head(8).values,
    "condition_rep": adata.obs[REPL_COL].head(8).values
}))

# ============================================================
# BASE OBS TABLE
# ============================================================
base_obs = adata.obs[[COND_CONSOL, COND_COL, REPL_COL, RUN_COL, STATE_COL]].copy()

# remove WT from DAM entirely
drop_mask = (
    (base_obs[STATE_COL].astype(str) == "DAM") &
    (base_obs[COND_CONSOL].astype(str).isin(wt_conds))
)
base_obs = base_obs.loc[~drop_mask].copy()

# apply same filtering back to adata so rows stay aligned
adata = adata[base_obs.index].copy()

print("\nCounts after removing WT from DAM:")
print(base_obs.groupby([STATE_COL, COND_CONSOL]).size())

present_conds = [c for c in cond_order if c in set(base_obs[COND_CONSOL].astype(str))]
extras = sorted(set(base_obs[COND_CONSOL].astype(str)) - set(present_conds))
present_conds = present_conds + extras

print("\npresent_conds:", present_conds)

plot_palette = {c: cond_palette.get(c, "#808080") for c in present_conds}
black_palette = {c: "black" for c in present_conds}

# ============================================================
# HELPERS
# ============================================================
def get_gene_counts(adata_obj, gene, layer=COUNTS_LAYER):
    X = adata_obj[:, gene].layers[layer]
    if sp.issparse(X):
        return np.asarray(X.toarray()).ravel()
    return np.asarray(X).ravel()

def clip_upper_percentile(values, pct=95, min_clip_value=None):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return arr, np.nan

    upper = np.percentile(arr[finite], pct)
    if min_clip_value is not None:
        upper = max(upper, min_clip_value)

    clipped = np.clip(arr, a_min=None, a_max=upper)
    return clipped, upper

def annotate_pct_expressing(ax, df_group, mean_col, y_top):
    """
    Place % expressing text just above each group's mean diamond.
    Uses seaborn's default dodge offsets for 4 hue levels.
    """
    hue_offsets = {
        4: [-0.30, -0.10, 0.10, 0.30],
        3: [-0.25, 0.00, 0.25],
        2: [-0.20, 0.20],
        1: [0.00],
    }
    offsets = hue_offsets.get(len(present_conds), np.linspace(-0.30, 0.30, len(present_conds)))

    state_to_x = {state: i for i, state in enumerate(state_order)}
    cond_to_offset = {cond: offsets[i] for i, cond in enumerate(present_conds)}

    y_pad = max(y_top * PCT_EXPR_OFFSET_FRAC, 0.03)

    for _, row in df_group.iterrows():
        state = str(row[STATE_COL])
        cond = str(row[COND_CONSOL])

        if state not in state_to_x or cond not in cond_to_offset:
            continue

        x = state_to_x[state] + cond_to_offset[cond]
        y = float(row[mean_col]) + y_pad
        pct = float(row["pct_expr"])

        # keep text inside plotting area
        y = min(y, y_top * 0.98)

        ax.text(
            x, y,
            f"{pct:.1f}%",
            ha="center", va="bottom",
            fontsize=PCT_EXPR_TEXT_SIZE,
            color="black",
            rotation=0
        )

def make_violin_page(
    df_plot,
    df_group,
    gene,
    y_col,
    mean_col,
    ylabel,
    title_suffix,
    pdf_handle
):
    plt.figure(figsize=(9.2, 6.0))
    ax = plt.gca()

    sns.violinplot(
        data=df_plot,
        x=STATE_COL,
        y=y_col,
        hue=COND_CONSOL,
        order=state_order,
        hue_order=present_conds,
        palette=plot_palette,
        cut=VIOLIN_CUT,
        density_norm="width",
        inner=None,
        linewidth=1,
        ax=ax
    )

    # ALL black dots, not subsampled
    sns.stripplot(
        data=df_plot,
        x=STATE_COL,
        y=y_col,
        hue=COND_CONSOL,
        order=state_order,
        hue_order=present_conds,
        dodge=True,
        jitter=POINT_JITTER,
        size=POINT_SIZE,
        alpha=POINT_ALPHA,
        linewidth=0,
        palette=black_palette,
        ax=ax
    )

    sns.stripplot(
        data=df_group,
        x=STATE_COL,
        y=mean_col,
        hue=COND_CONSOL,
        order=state_order,
        hue_order=present_conds,
        dodge=True,
        marker="D",
        size=7,
        edgecolor="black",
        linewidth=0.8,
        palette=plot_palette,
        ax=ax
    )

    vals = df_plot[y_col].values
    if len(vals) == 0 or not np.any(np.isfinite(vals)):
        ymax = 1.0
    else:
        if "clip" in y_col:
            ymax = np.percentile(vals[np.isfinite(vals)], 98)
        else:
            ymax = np.percentile(vals[np.isfinite(vals)], 99)

        if not np.isfinite(ymax) or ymax <= 0:
            ymax = float(np.nanmax(vals[np.isfinite(vals)])) if np.any(np.isfinite(vals)) else 1.0
        if not np.isfinite(ymax) or ymax <= 0:
            ymax = 1.0

    y_top = ymax * 1.10
    ax.set_ylim(0, y_top)

    annotate_pct_expressing(ax, df_group, mean_col=mean_col, y_top=y_top)

    ax.set_title(f"{gene}\n{title_suffix}")
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)

    if ax.legend_ is not None:
        ax.legend_.remove()

    legend_handles = [
        Patch(facecolor=plot_palette[c], edgecolor="black", label=c)
        for c in present_conds
    ]
    ax.legend(
        handles=legend_handles,
        title="Condition",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False
    )

    plt.tight_layout()
    pdf_handle.savefig(ax.figure, dpi=200)
    plt.close()

# ============================================================
# LOOP GENES
# ============================================================
genes = adata.var_names.to_list()
print("\nn_genes:", len(genes))
print("Writing RAW PDF:", OUT_PDF_RAW)
print("Writing LOG1P PDF:", OUT_PDF_LOG1P)
print("Writing LOG1P CLIP95 PDF:", OUT_PDF_LOG1P_CLIP95)

all_cells_long = []
all_group_summaries = []
all_pvals = []

sns.set_style("whitegrid")

with PdfPages(OUT_PDF_RAW) as pdf_raw, \
     PdfPages(OUT_PDF_LOG1P) as pdf_log1p, \
     PdfPages(OUT_PDF_LOG1P_CLIP95) as pdf_log1p_clip95:

    for i, gene in enumerate(genes, start=1):
        expr_full = get_gene_counts(adata, gene, layer=COUNTS_LAYER)

        # row-aligned per-cell table
        df = base_obs.copy()
        idx = adata.obs_names.get_indexer(df.index)
        if np.any(idx < 0):
            raise ValueError(f"Index alignment failed for gene {gene}")

        df["expr"] = expr_full[idx]
        df["expr_log1p"] = np.log1p(df["expr"])
        df["expr_log1p_clip95"], clip_upper_log1p = clip_upper_percentile(
            df["expr_log1p"].values,
            pct=CLIP_PERCENTILE,
            min_clip_value=MIN_CLIP_VALUE
        )
        df["clip_upper_log1p_95"] = clip_upper_log1p
        df["expr_binary"] = (df["expr"] > 0).astype(int)
        df["gene"] = gene

        # save per-cell long data
        all_cells_long.append(df.copy())

        # group summaries from all cells
        df_group = (
            df.groupby([COND_CONSOL, STATE_COL], observed=True)
              .agg(
                  mean_expr=("expr", "mean"),
                  median_expr=("expr", "median"),
                  mean_expr_log1p=("expr_log1p", "mean"),
                  median_expr_log1p=("expr_log1p", "median"),
                  mean_expr_log1p_clip95=("expr_log1p_clip95", "mean"),
                  median_expr_log1p_clip95=("expr_log1p_clip95", "median"),
                  n_cells=("expr", "size"),
                  n_expr=("expr_binary", "sum"),
                  pct_expr=("expr_binary", lambda x: 100.0 * np.mean(x)),
              )
              .reset_index()
        )
        df_group["gene"] = gene
        df_group["clip_upper_log1p_95"] = clip_upper_log1p
        all_group_summaries.append(df_group)

        # Welch p-tests on ALL CELLS
        for state in state_order:
            sub = df[df[STATE_COL].astype(str) == state].copy()
            if sub.empty:
                continue

            for cond_a, cond_b in COMPARISONS:
                a = sub.loc[sub[COND_CONSOL].astype(str) == cond_a, "expr"].dropna().to_numpy()
                b = sub.loc[sub[COND_CONSOL].astype(str) == cond_b, "expr"].dropna().to_numpy()

                a_log1p = sub.loc[sub[COND_CONSOL].astype(str) == cond_a, "expr_log1p"].dropna().to_numpy()
                b_log1p = sub.loc[sub[COND_CONSOL].astype(str) == cond_b, "expr_log1p"].dropna().to_numpy()

                a_log1p_clip95 = sub.loc[
                    sub[COND_CONSOL].astype(str) == cond_a, "expr_log1p_clip95"
                ].dropna().to_numpy()
                b_log1p_clip95 = sub.loc[
                    sub[COND_CONSOL].astype(str) == cond_b, "expr_log1p_clip95"
                ].dropna().to_numpy()

                if len(a) == 0 or len(b) == 0:
                    continue

                try:
                    stat_raw, pval_raw = ttest_ind(a, b, equal_var=False)
                except Exception:
                    stat_raw, pval_raw = (np.nan, np.nan)

                try:
                    stat_log1p, pval_log1p = ttest_ind(a_log1p, b_log1p, equal_var=False)
                except Exception:
                    stat_log1p, pval_log1p = (np.nan, np.nan)

                try:
                    stat_log1p_clip95, pval_log1p_clip95 = ttest_ind(
                        a_log1p_clip95, b_log1p_clip95, equal_var=False
                    )
                except Exception:
                    stat_log1p_clip95, pval_log1p_clip95 = (np.nan, np.nan)

                mean_a = float(np.mean(a))
                mean_b = float(np.mean(b))

                mean_a_log1p = float(np.mean(a_log1p))
                mean_b_log1p = float(np.mean(b_log1p))

                mean_a_log1p_clip95 = float(np.mean(a_log1p_clip95))
                mean_b_log1p_clip95 = float(np.mean(b_log1p_clip95))

                pct_expr_a = float(100.0 * np.mean(a > 0)) if len(a) > 0 else np.nan
                pct_expr_b = float(100.0 * np.mean(b > 0)) if len(b) > 0 else np.nan

                all_pvals.append({
                    "gene": gene,
                    "state": state,
                    "cond_a": cond_a,
                    "cond_b": cond_b,
                    "comparison": f"{cond_a}_vs_{cond_b}",

                    "mean_expr_a": mean_a,
                    "mean_expr_b": mean_b,
                    "median_expr_a": float(np.median(a)),
                    "median_expr_b": float(np.median(b)),
                    "pct_expr_a": pct_expr_a,
                    "pct_expr_b": pct_expr_b,
                    "log2FC_allCells": float(np.log2((mean_a + PSEUDOCOUNT) / (mean_b + PSEUDOCOUNT))),
                    "welch_t_raw": stat_raw,
                    "pval_raw": pval_raw,

                    "mean_expr_log1p_a": mean_a_log1p,
                    "mean_expr_log1p_b": mean_b_log1p,
                    "median_expr_log1p_a": float(np.median(a_log1p)),
                    "median_expr_log1p_b": float(np.median(b_log1p)),
                    "diff_mean_log1p_a_minus_b": float(mean_a_log1p - mean_b_log1p),
                    "welch_t_log1p": stat_log1p,
                    "pval_log1p": pval_log1p,

                    "mean_expr_log1p_clip95_a": mean_a_log1p_clip95,
                    "mean_expr_log1p_clip95_b": mean_b_log1p_clip95,
                    "median_expr_log1p_clip95_a": float(np.median(a_log1p_clip95)),
                    "median_expr_log1p_clip95_b": float(np.median(b_log1p_clip95)),
                    "diff_mean_log1p_clip95_a_minus_b": float(mean_a_log1p_clip95 - mean_b_log1p_clip95),
                    "welch_t_log1p_clip95": stat_log1p_clip95,
                    "pval_log1p_clip95": pval_log1p_clip95,

                    "clip_upper_log1p_95": clip_upper_log1p,
                    "n_cells_a": int(len(a)),
                    "n_cells_b": int(len(b)),
                    "test_level": "all_cells",
                    "counts_layer": COUNTS_LAYER,
                })

        # 1) RAW page
        make_violin_page(
            df_plot=df,
            df_group=df_group,
            gene=gene,
            y_col="expr",
            mean_col="mean_expr",
            ylabel="Raw counts per cell",
            title_suffix=(
                "% expressing shown above each violin\n"
                "violin=all cells; black dots=all cells; diamonds=mean across cells"
            ),
            pdf_handle=pdf_raw
        )

        # 2) LOG1P page
        make_violin_page(
            df_plot=df,
            df_group=df_group,
            gene=gene,
            y_col="expr_log1p",
            mean_col="mean_expr_log1p",
            ylabel="log1p(counts per cell)",
            title_suffix=(
                "% expressing shown above each violin\n"
                "violin=all cells; black dots=all cells; diamonds=mean across cells"
            ),
            pdf_handle=pdf_log1p
        )

        # 3) LOG1P CLIPPED page
        make_violin_page(
            df_plot=df,
            df_group=df_group,
            gene=gene,
            y_col="expr_log1p_clip95",
            mean_col="mean_expr_log1p_clip95",
            ylabel=f"log1p(counts per cell), clipped at {CLIP_PERCENTILE}th percentile",
            title_suffix=(
                f"% expressing shown above each violin\n"
                f"log1p(counts per cell), upper-clipped at gene-wise {CLIP_PERCENTILE}th percentile\n"
                "violin=all cells; black dots=all cells; diamonds=mean across cells"
            ),
            pdf_handle=pdf_log1p_clip95
        )

        if i % 50 == 0:
            print(f"...{i}/{len(genes)} pages written")

print("Finished RAW PDF:", OUT_PDF_RAW)
print("Finished LOG1P PDF:", OUT_PDF_LOG1P)
print("Finished LOG1P CLIP95 PDF:", OUT_PDF_LOG1P_CLIP95)

# ============================================================
# EXPORT CSVs
# ============================================================
all_cells_long_df = pd.concat(all_cells_long, ignore_index=True)
all_group_summary_df = pd.concat(all_group_summaries, ignore_index=True)
all_pvals_df = pd.DataFrame(all_pvals)

all_cells_long_df.to_csv(OUT_CELL_CSV, index=False)
all_group_summary_df.to_csv(OUT_GROUP_CSV, index=False)
all_pvals_df.to_csv(OUT_PVAL_CSV, index=False)

print("Wrote per-cell long table:", OUT_CELL_CSV)
print("Wrote group summaries:", OUT_GROUP_CSV)
print("Wrote Welch p-values:", OUT_PVAL_CSV)
print("Done.")

adata shape after filtering: (48534, 347)
State counts:
miranda_cell_type_refined_combinedUpdate
Homeostatic microglia    34101
DAM                      14433
Name: count, dtype: int64
Condition counts:
condition_consol
FAD-Veh    18151
FAD-Abe    15479
WT-Veh      8135
WT-Abe      6769
Name: count, dtype: int64

Example condition -> condition_rep:
  condition condition_rep
0  FAD-Abe3      FAD-Abe3
1  FAD-Abe3      FAD-Abe3
2  FAD-Abe3      FAD-Abe3
3  FAD-Abe3      FAD-Abe3
4  FAD-Abe3      FAD-Abe3
5  FAD-Abe3      FAD-Abe3
6  FAD-Abe3      FAD-Abe3
7  FAD-Abe3      FAD-Abe3

Counts after removing WT from DAM:
miranda_cell_type_refined_combinedUpdate  condition_consol
DAM                                       WT-Veh                  0
                                          WT-Abe                  0
                                          FAD-Veh              7447
                                          FAD-Abe              6634
Homeostatic microglia                     WT-Veh

/tmp/ipykernel_3780948/3510803491.py:182: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(base_obs.groupby([STATE_COL, COND_CONSOL]).size())
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` p

...50/347 pages written


/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_

...100/347 pages written


...

...250/347 pages written


/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_

...300/347 pages written


/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_3780948/3510803491.py:269: FutureWarning: 

The `scale` parameter has been renamed and will be removed in v0.15.0. Pass `density_norm='width'` for the same effect.
  sns.violinplot(
/tmp/ipykernel_

Finished RAW PDF: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/geneCounts_allCells_violinplots_from_ad_full_palette2_pctExpr_ALLDOTS/ALLgenes_countsPerCell_allCells_violin_RAW_pctExpr_ALLDOTS.pdf
Finished LOG1P PDF: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/geneCounts_allCells_violinplots_from_ad_full_palette2_pctExpr_ALLDOTS/ALLgenes_countsPerCell_allCells_violin_LOG1P_pctExpr_ALLDOTS.pdf
Finished LOG1P CLIP95 PDF: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/geneCounts_allCells_violinplots_from_ad_full_palette2_pctExpr_ALLDOTS/ALLgenes_countsPerCell_allCells_violin_LOG1P_CLIP95_pctExpr_ALLDOTS.pdf
Wrote per-cell long table: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/geneCounts_allCells_violinplots_from_ad_full_palette2_pctExpr_ALLDOTS/ALLgenes_allCells_long_pctExpr.csv
Wrote group summaries: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/geneCounts_allC

In [20]:
# ============================================================
# MARKER-ONLY VIOLIN PLOTS BY REPLICATE-LEVEL CONDITION
#
# - ONLY plots the requested marker genes
# - Groups by replicate-level `condition`
#   (e.g. FAD-Veh1, FAD-Veh2, WT-Veh1, etc. each gets its own violin)
# - Keeps BOTH states on the same page:
#     x-axis = DAM / Homeostatic microglia
#     hue    = replicate-level condition
# - Uses ALL CELLS and raw counts from adata.layers["counts"]
# - Adds percent expressing (% cells with expr > 0)
# - Plots ALL black dots
# - WRITES ONLY:
#     1) RAW
#     2) LOG1P
# - DOES NOT write LOG1P_CLIP95
# ============================================================

import os
import re
import colorsys
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Patch

# ============================================================
# PATHS / SETTINGS
# ============================================================
ADATA_PATH = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/combined/"
    "sub_adata_microglia_leiden6_noOutside_reclustered_gatex2_with_miranda_"
    "DAMvHOME_damifnhome_combined_withCounts_combinedcleaned_UPDATE_palette2.h5ad"
)

# keep same folder as before
OUT_DIR = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/"
    "geneCounts_allCells_violinplots_from_ad_full_palette2_pctExpr_ALLDOTS_"
    "consolPlusConditionMarkers/markerGenes_byCondition"
)
os.makedirs(OUT_DIR, exist_ok=True)

# overwrite these outputs
OUT_PDF_RAW = os.path.join(
    OUT_DIR,
    "MARKERgenes_allCells_violin_byCondition_pctExpr_ALLDOTS_RAW.pdf"
)
OUT_PDF_LOG1P = os.path.join(
    OUT_DIR,
    "MARKERgenes_allCells_violin_byCondition_pctExpr_ALLDOTS_LOG1P.pdf"
)

OUT_GROUP_CSV = os.path.join(
    OUT_DIR,
    "MARKERgenes_groupSummary_byCondition_long_pctExpr.csv"
)
OUT_MARKER_STATUS_TXT = os.path.join(
    OUT_DIR,
    "marker_gene_presence_report.txt"
)

STATE_COL = "miranda_cell_type_refined_combinedUpdate"
COND_COL = "condition"
COND_CONSOL = "condition_consol"
RUN_COL = "run_key"
LEIDEN_COL = "leiden"
COUNTS_LAYER = "counts"

RESTRICT_LEIDEN = False
LEIDEN_TARGET = "6"

state_order = ["DAM", "Homeostatic microglia"]
keep_states = set(state_order)

# keep prior DAM exclusion logic
wt_conds = {"WT-Veh", "WT-Abe"}

# consolidated family colors
cond_consol_palette = {
    "FAD-Abe": "#1f77b4",
    "FAD-Veh": "#ff7f0e",
    "WT-Abe":  "#2ca02c",
    "WT-Veh":  "#d62728",
}

# violin settings
VIOLIN_CUT = 0
VIOLIN_SCALE = "width"
POINT_SIZE = 0.55
POINT_ALPHA = 0.20
POINT_JITTER = 0.18

# annotation settings
PCT_EXPR_TEXT_SIZE = 7
PCT_EXPR_OFFSET_FRAC = 0.025

# ============================================================
# MARKER LISTS (ONLY THESE WILL BE PLOTTED)
# ============================================================
MARKERS_UP = [
    "Acsbg1", "Gfap", "Axl", "Rmst", "Apoe", "H2-DMa", "Cntn6", "Fcgr1",
    "Cst7", "Cux2", "Fign", "Pdcd1", "Aqp4", "Dpy19l1", "Ccr5",
    "Trem2", "Slc11a1", "Pag1", "Olfml3"
]

MARKERS_DOWN = [
    "Arc", "Sall1", "Mertk", "Cd9", "Cd52", "Foxp1", "Inpp4b", "Hpcal1",
    "Jun", "Nrn1", "Egr1", "2010300C02Rik", "Lpl", "Nrp2", "Serpine2",
    "Gpnmb", "Lgals3", "Wfs1", "Tgfbr1", "Gas7", "Meis2", "Calb1",
    "Havcr2", "Mafb", "Rasgrf2", "Spi1", "Cxcr4", "Tyrobp", "Fabp5",
    "Ctsb", "Tmem163", "Lrpap1", "Crybb1", "Laptm5", "Siglech", "Hexb",
    "Mef2a", "Kctd12", "Rhob", "P2ry12", "Ikzf1", "Ctsd", "Apbb2",
    "Ctsz", "Cd68", "Lat2", "Cx3cr1", "Runx1"
]

MARKER_GENES = list(dict.fromkeys(MARKERS_UP + MARKERS_DOWN))

# ============================================================
# HELPERS
# ============================================================
def get_gene_counts(adata_obj, gene, layer=COUNTS_LAYER):
    X = adata_obj[:, gene].layers[layer]
    if sp.issparse(X):
        return np.asarray(X.toarray()).ravel()
    return np.asarray(X).ravel()

def lighten_hex(hex_color, amount=0.25):
    hex_color = hex_color.lstrip("#")
    r = int(hex_color[0:2], 16) / 255.0
    g = int(hex_color[2:4], 16) / 255.0
    b = int(hex_color[4:6], 16) / 255.0

    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = min(1.0, l + amount * (1 - l))
    r2, g2, b2 = colorsys.hls_to_rgb(h, l, s)

    return "#{:02x}{:02x}{:02x}".format(
        int(round(r2 * 255)),
        int(round(g2 * 255)),
        int(round(b2 * 255)),
    )

def build_condition_palette(groups):
    """
    Replicate-level colors preserving consolidated family color.
    Example:
      FAD-Abe1, FAD-Abe2 -> shades of blue
      FAD-Veh1, FAD-Veh2 -> shades of orange
    """
    family_to_base = {
        "FAD-Abe": cond_consol_palette["FAD-Abe"],
        "FAD-Veh": cond_consol_palette["FAD-Veh"],
        "WT-Abe": cond_consol_palette["WT-Abe"],
        "WT-Veh": cond_consol_palette["WT-Veh"],
    }

    fam_to_members = {}
    for g in groups:
        fam = re.sub(r"\d+$", "", str(g))
        fam_to_members.setdefault(fam, []).append(g)

    palette = {}
    for fam, members in fam_to_members.items():
        members = sorted(members)
        base = family_to_base.get(fam, "#808080")

        if len(members) == 1:
            palette[members[0]] = base
        else:
            amounts = np.linspace(0.00, 0.35, len(members))
            for member, amt in zip(members, amounts):
                palette[member] = lighten_hex(base, amount=float(amt))

    return palette

def annotate_pct_expressing(ax, df_group, mean_col, y_top, state_order, hue_order):
    hue_offsets_map = {
        1: [0.00],
        2: [-0.20, 0.20],
        3: [-0.25, 0.00, 0.25],
        4: [-0.30, -0.10, 0.10, 0.30],
        5: [-0.34, -0.17, 0.00, 0.17, 0.34],
        6: [-0.36, -0.22, -0.08, 0.08, 0.22, 0.36],
        7: [-0.38, -0.25, -0.13, 0.00, 0.13, 0.25, 0.38],
        8: [-0.40, -0.29, -0.17, -0.06, 0.06, 0.17, 0.29, 0.40],
        9: [-0.42, -0.31, -0.21, -0.10, 0.00, 0.10, 0.21, 0.31, 0.42],
        10: [-0.44, -0.34, -0.24, -0.15, -0.05, 0.05, 0.15, 0.24, 0.34, 0.44],
        11: [-0.45, -0.36, -0.27, -0.18, -0.09, 0.00, 0.09, 0.18, 0.27, 0.36, 0.45],
        12: [-0.46, -0.38, -0.29, -0.21, -0.13, -0.04, 0.04, 0.13, 0.21, 0.29, 0.38, 0.46],
    }

    if len(hue_order) in hue_offsets_map:
        offsets = hue_offsets_map[len(hue_order)]
    else:
        offsets = np.linspace(-0.46, 0.46, len(hue_order))

    state_to_x = {state: i for i, state in enumerate(state_order)}
    hue_to_offset = {h: offsets[i] for i, h in enumerate(hue_order)}

    y_pad = max(y_top * PCT_EXPR_OFFSET_FRAC, 0.03)

    for _, row in df_group.iterrows():
        state = str(row[STATE_COL])
        hue = str(row[COND_COL])

        if state not in state_to_x or hue not in hue_to_offset:
            continue

        x = state_to_x[state] + hue_to_offset[hue]
        y = float(row[mean_col]) + y_pad
        pct = float(row["pct_expr"])
        y = min(y, y_top * 0.98)

        ax.text(
            x, y, f"{pct:.1f}%",
            ha="center", va="bottom",
            fontsize=PCT_EXPR_TEXT_SIZE,
            color="black"
        )

def make_violin_page(
    df_plot,
    df_group,
    gene,
    y_col,
    mean_col,
    ylabel,
    title_suffix,
    pdf_handle,
    hue_order,
    palette
):
    plt.figure(figsize=(max(11.5, 1.25 * len(hue_order) + 6.5), 6.0))
    ax = plt.gca()

    sns.violinplot(
        data=df_plot,
        x=STATE_COL,
        y=y_col,
        hue=COND_COL,
        order=state_order,
        hue_order=hue_order,
        palette=palette,
        cut=VIOLIN_CUT,
        density_norm="width",
        inner=None,
        linewidth=1,
        ax=ax
    )

    sns.stripplot(
        data=df_plot,
        x=STATE_COL,
        y=y_col,
        hue=COND_COL,
        order=state_order,
        hue_order=hue_order,
        dodge=True,
        jitter=POINT_JITTER,
        size=POINT_SIZE,
        alpha=POINT_ALPHA,
        linewidth=0,
        palette={k: "black" for k in hue_order},
        ax=ax
    )

    sns.stripplot(
        data=df_group,
        x=STATE_COL,
        y=mean_col,
        hue=COND_COL,
        order=state_order,
        hue_order=hue_order,
        dodge=True,
        marker="D",
        size=7,
        edgecolor="black",
        linewidth=0.8,
        palette=palette,
        ax=ax
    )

    vals = df_plot[y_col].values
    finite_vals = vals[np.isfinite(vals)]

    if len(finite_vals) == 0:
        ymax = 1.0
    else:
        ymax = np.percentile(finite_vals, 99)
        if not np.isfinite(ymax) or ymax <= 0:
            ymax = float(np.nanmax(finite_vals))
        if not np.isfinite(ymax) or ymax <= 0:
            ymax = 1.0

    y_top = ymax * 1.10
    ax.set_ylim(0, y_top)

    annotate_pct_expressing(
        ax=ax,
        df_group=df_group,
        mean_col=mean_col,
        y_top=y_top,
        state_order=state_order,
        hue_order=hue_order
    )

    ax.set_title(f"{gene}\n{title_suffix}")
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)

    if ax.legend_ is not None:
        ax.legend_.remove()

    legend_handles = [
        Patch(facecolor=palette[h], edgecolor="black", label=h)
        for h in hue_order
    ]
    ax.legend(
        handles=legend_handles,
        title="Condition (replicate-level)",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False
    )

    plt.tight_layout()
    pdf_handle.savefig(ax.figure, dpi=200)
    plt.close()

# ============================================================
# LOAD OBJECT
# ============================================================
adata = sc.read_h5ad(ADATA_PATH)

if RESTRICT_LEIDEN:
    adata = adata[adata.obs[LEIDEN_COL].astype(str) == str(LEIDEN_TARGET)].copy()

if COUNTS_LAYER not in adata.layers:
    raise KeyError(
        f"adata.layers['{COUNTS_LAYER}'] not found. "
        f"Available layers: {list(adata.layers.keys())}"
    )

for col in [STATE_COL, COND_COL, COND_CONSOL]:
    if col not in adata.obs.columns:
        raise KeyError(f"Missing required obs column: {col}")

adata = adata[adata.obs[STATE_COL].astype(str).isin(keep_states)].copy()
adata.obs[STATE_COL] = (
    adata.obs[STATE_COL]
    .astype(str)
    .astype("category")
    .cat.remove_unused_categories()
)

print("adata shape after filtering:", adata.shape)
print("State counts:")
print(adata.obs[STATE_COL].value_counts())
print("Condition_consol counts:")
print(adata.obs[COND_CONSOL].value_counts())
print("Condition counts:")
print(adata.obs[COND_COL].value_counts())

# ============================================================
# BASE OBS TABLE
# ============================================================
base_obs = adata.obs[[COND_CONSOL, COND_COL, RUN_COL, STATE_COL]].copy()

# remove WT from DAM entirely using consolidated condition logic
drop_mask = (
    (base_obs[STATE_COL].astype(str) == "DAM") &
    (base_obs[COND_CONSOL].astype(str).isin(wt_conds))
)
base_obs = base_obs.loc[~drop_mask].copy()

# keep adata aligned to filtered obs
adata = adata[base_obs.index].copy()

print("\nCounts after removing WT from DAM:")
print(base_obs.groupby([STATE_COL, COND_COL]).size())

# ============================================================
# MARKER PRESENCE CHECK
# ============================================================
genes_all = adata.var_names.to_list()
marker_present = [g for g in MARKER_GENES if g in genes_all]
marker_missing = [g for g in MARKER_GENES if g not in genes_all]

with open(OUT_MARKER_STATUS_TXT, "w") as f:
    f.write("Marker genes requested:\n")
    f.write("\n".join(MARKER_GENES))
    f.write("\n\nPresent in adata:\n")
    f.write("\n".join(marker_present))
    f.write("\n\nMissing from adata:\n")
    f.write("\n".join(marker_missing))

print(f"\nMarker genes present: {len(marker_present)} / {len(MARKER_GENES)}")
if marker_missing:
    print("Missing markers:", marker_missing)

# ============================================================
# ORDERS / PALETTE
# ============================================================
present_conditions = sorted(base_obs[COND_COL].astype(str).unique().tolist())
plot_palette_condition = build_condition_palette(present_conditions)

print("\npresent_conditions:", present_conditions)
print("Writing RAW PDF:", OUT_PDF_RAW)
print("Writing LOG1P PDF:", OUT_PDF_LOG1P)

# ============================================================
# LOOP MARKER GENES ONLY
# ============================================================
sns.set_style("whitegrid")

all_group_summaries = []

with PdfPages(OUT_PDF_RAW) as pdf_raw, \
     PdfPages(OUT_PDF_LOG1P) as pdf_log1p:

    for i, gene in enumerate(marker_present, start=1):
        expr_full = get_gene_counts(adata, gene, layer=COUNTS_LAYER)

        df = base_obs.copy()
        idx = adata.obs_names.get_indexer(df.index)
        if np.any(idx < 0):
            raise ValueError(f"Index alignment failed for gene {gene}")

        df["expr"] = expr_full[idx]
        df["expr_log1p"] = np.log1p(df["expr"])
        df["expr_binary"] = (df["expr"] > 0).astype(int)
        df["gene"] = gene

        df_group = (
            df.groupby([COND_COL, STATE_COL], observed=True)
              .agg(
                  mean_expr=("expr", "mean"),
                  median_expr=("expr", "median"),
                  mean_expr_log1p=("expr_log1p", "mean"),
                  median_expr_log1p=("expr_log1p", "median"),
                  n_cells=("expr", "size"),
                  n_expr=("expr_binary", "sum"),
                  pct_expr=("expr_binary", lambda x: 100.0 * np.mean(x)),
              )
              .reset_index()
        )
        df_group["gene"] = gene
        all_group_summaries.append(df_group)

        base_title = (
            "% expressing shown above each violin\n"
            "violin=all cells; black dots=all cells; diamonds=mean across cells\n"
            "grouped by replicate-level condition"
        )

        make_violin_page(
            df_plot=df,
            df_group=df_group,
            gene=gene,
            y_col="expr",
            mean_col="mean_expr",
            ylabel="Raw counts per cell",
            title_suffix=base_title,
            pdf_handle=pdf_raw,
            hue_order=present_conditions,
            palette=plot_palette_condition
        )

        make_violin_page(
            df_plot=df,
            df_group=df_group,
            gene=gene,
            y_col="expr_log1p",
            mean_col="mean_expr_log1p",
            ylabel="log1p(counts per cell)",
            title_suffix=base_title,
            pdf_handle=pdf_log1p,
            hue_order=present_conditions,
            palette=plot_palette_condition
        )

        if i % 10 == 0:
            print(f"...{i}/{len(marker_present)} marker genes written")

# ============================================================
# EXPORT GROUP SUMMARY
# ============================================================
all_group_summary_df = pd.concat(all_group_summaries, ignore_index=True)
all_group_summary_df.to_csv(OUT_GROUP_CSV, index=False)

print("Finished RAW PDF:", OUT_PDF_RAW)
print("Finished LOG1P PDF:", OUT_PDF_LOG1P)
print("Wrote group summary:", OUT_GROUP_CSV)
print("Wrote marker presence report:", OUT_MARKER_STATUS_TXT)
print("Done.")

adata shape after filtering: (48534, 347)
State counts:
miranda_cell_type_refined_combinedUpdate
Homeostatic microglia    34101
DAM                      14433
Name: count, dtype: int64
Condition_consol counts:
condition_consol
FAD-Veh    18151
FAD-Abe    15479
WT-Veh      8135
WT-Abe      6769
Name: count, dtype: int64
Condition counts:
condition
FAD-Veh2    7782
FAD-Abe2    6495
FAD-Veh1    6318
FAD-Abe3    4519
FAD-Abe1    4465
FAD-Veh3    4051
WT-Veh2     3556
WT-Veh1     2710
WT-Abe1     2356
WT-Abe3     2216
WT-Abe2     2197
WT-Veh3     1869
Name: count, dtype: int64

Counts after removing WT from DAM:
miranda_cell_type_refined_combinedUpdate  condition
DAM                                       FAD-Abe1     1670
                                          FAD-Abe2     2852
                                          FAD-Abe3     2112
                                          FAD-Veh1     2762
                                          FAD-Veh2     3219
                                 

/tmp/ipykernel_3780948/3076312323.py:386: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(base_obs.groupby([STATE_COL, COND_COL]).size())


...10/67 marker genes written
...20/67 marker genes written
...30/67 marker genes written
...40/67 marker genes written
...50/67 marker genes written
...60/67 marker genes written
Finished RAW PDF: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/geneCounts_allCells_violinplots_from_ad_full_palette2_pctExpr_ALLDOTS_consolPlusConditionMarkers/markerGenes_byCondition/MARKERgenes_allCells_violin_byCondition_pctExpr_ALLDOTS_RAW.pdf
Finished LOG1P PDF: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/geneCounts_allCells_violinplots_from_ad_full_palette2_pctExpr_ALLDOTS_consolPlusConditionMarkers/markerGenes_byCondition/MARKERgenes_allCells_violin_byCondition_pctExpr_ALLDOTS_LOG1P.pdf
Wrote group summary: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/geneCounts_allCells_violinplots_from_ad_full_palette2_pctExpr_ALLDOTS_consolPlusConditionMarkers/markerGenes_byCondition/MARKERgenes_groupSummary_byCondition_long_pctExpr.csv
W

In [35]:
# ============================================================
# MARKER DOT PLOTS
#
# OUTPUTS:
#   Two color-metric versions:
#
#   1) populationMean:
#      dot size  = % expressing
#      dot color = mean raw counts across ALL cells in group
#
#   2) expressingOnlyMean:
#      dot size  = % expressing
#      dot color = mean raw counts among expressing cells only
#
#   Split by:
#      - group_col: condition_consol vs condition
#      - state: DAM vs Homeostatic microglia
#      - gene set: UP vs DOWN
#
# So total outputs = 16 PDFs
#
# NOTES:
# - Uses ALL cells
# - Uses raw counts from adata.layers["counts"]
# - Restricts to requested marker genes only
# - Removes WT from DAM using condition_consol logic
# - Uses state-specific row orders so blank WT rows do not appear in DAM
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import scipy.sparse as sp

# ============================================================
# PATHS / SETTINGS
# ============================================================
ADATA_PATH = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/combined/"
    "sub_adata_microglia_leiden6_noOutside_reclustered_gatex2_with_miranda_"
    "DAMvHOME_damifnhome_combined_withCounts_combinedcleaned_UPDATE_palette2.h5ad"
)

OUT_DIR = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/"
    "marker_dotplots_condition_and_consol_UPDOWN_twoColorModes"
)
os.makedirs(OUT_DIR, exist_ok=True)

STATE_COL = "miranda_cell_type_refined_combinedUpdate"
COND_COL = "condition"
COND_CONSOL = "condition_consol"
RUN_COL = "run_key"
LEIDEN_COL = "leiden"
COUNTS_LAYER = "counts"

RESTRICT_LEIDEN = False
LEIDEN_TARGET = "6"

state_order = ["DAM", "Homeostatic microglia"]
keep_states = set(state_order)
wt_conds = {"WT-Veh", "WT-Abe"}

# dot plot appearance
FIG_DPI = 220
SIZE_MIN = 20
SIZE_MAX = 420
CMAP = "RdBu_r"

# preferred ordering for consolidated groups
cond_consol_order = ["FAD-Abe", "FAD-Veh", "WT-Abe", "WT-Veh"]

# ============================================================
# MARKER LISTS
# ============================================================
MARKERS_UP = [
    "Acsbg1", "Gfap", "Axl", "Rmst", "Apoe", "H2-DMa", "Cntn6", "Fcgr1",
    "Cst7", "Cux2", "Fign", "Pdcd1", "Aqp4", "Dpy19l1", "Ccr5",
    "Trem2", "Slc11a1", "Pag1", "Olfml3"
]

MARKERS_DOWN = [
    "Arc", "Sall1", "Mertk", "Cd9", "Cd52", "Foxp1", "Inpp4b", "Hpcal1",
    "Jun", "Nrn1", "Egr1", "2010300C02Rik", "Lpl", "Nrp2", "Serpine2",
    "Gpnmb", "Lgals3", "Wfs1", "Tgfbr1", "Gas7", "Meis2", "Calb1",
    "Havcr2", "Mafb", "Rasgrf2", "Spi1", "Cxcr4", "Tyrobp", "Fabp5",
    "Ctsb", "Tmem163", "Lrpap1", "Crybb1", "Laptm5", "Siglech", "Hexb",
    "Mef2a", "Kctd12", "Rhob", "P2ry12", "Ikzf1", "Ctsd", "Apbb2",
    "Ctsz", "Cd68", "Lat2", "Cx3cr1", "Runx1"
]

MARKER_GENES = list(dict.fromkeys(MARKERS_UP + MARKERS_DOWN))

# ============================================================
# HELPERS
# ============================================================
def get_gene_counts(adata_obj, gene, layer=COUNTS_LAYER):
    X = adata_obj[:, gene].layers[layer]
    if sp.issparse(X):
        return np.asarray(X.toarray()).ravel()
    return np.asarray(X).ravel()

def family_sort_key(cond_name):
    """
    Sort replicate-level conditions like:
      FAD-Abe1, FAD-Abe2, FAD-Abe3, FAD-Veh1, ...
    """
    cond_name = str(cond_name)
    m = re.match(r"^(.*?)(\d+)$", cond_name)
    if m:
        prefix, num = m.group(1), int(m.group(2))
    else:
        prefix, num = cond_name, 0

    prefix_rank = {
        "FAD-Abe": 0,
        "FAD-Veh": 1,
        "WT-Abe": 2,
        "WT-Veh": 3
    }.get(prefix, 999)

    return (prefix_rank, prefix, num, cond_name)

def pct_to_size(pct, size_min=SIZE_MIN, size_max=SIZE_MAX):
    pct = np.asarray(pct, dtype=float)
    return size_min + (pct / 100.0) * (size_max - size_min)

def summarize_gene_group(df_base, adata_obj, genes, group_col, state_value):
    """
    Returns long summary table with:
      gene
      group_col
      STATE_COL
      mean_expr_all       = mean raw counts across all cells
      mean_expr_pos       = mean raw counts among expressing cells only
      mean_expr_log1p_all = optional reference metric
      pct_expr            = percent expr > 0
      n_cells
      n_expr
    """
    sub_obs = df_base[df_base[STATE_COL].astype(str) == state_value].copy()
    results = []

    if sub_obs.empty:
        return pd.DataFrame(columns=[
            "gene", group_col, STATE_COL,
            "mean_expr_all", "mean_expr_pos", "mean_expr_log1p_all",
            "pct_expr", "n_cells", "n_expr"
        ])

    idx = adata_obj.obs_names.get_indexer(sub_obs.index)
    if np.any(idx < 0):
        raise ValueError(f"Index alignment failed for state {state_value}")

    for gene in genes:
        expr_full = get_gene_counts(adata_obj, gene, layer=COUNTS_LAYER)
        expr = expr_full[idx]
        expr_log1p = np.log1p(expr)

        tmp = sub_obs[[group_col]].copy()
        tmp["expr"] = expr
        tmp["expr_log1p"] = expr_log1p
        tmp["expr_binary"] = (expr > 0).astype(int)

        grp = (
            tmp.groupby(group_col, observed=True)
               .agg(
                   mean_expr_all=("expr", "mean"),
                   mean_expr_log1p_all=("expr_log1p", "mean"),
                   pct_expr=("expr_binary", lambda x: 100.0 * np.mean(x)),
                   n_cells=("expr", "size"),
                   n_expr=("expr_binary", "sum")
               )
               .reset_index()
        )

        # mean among expressing cells only
        expr_pos_means = []
        for group_value in grp[group_col].tolist():
            vals = tmp.loc[tmp[group_col] == group_value, "expr"].to_numpy()
            vals_pos = vals[vals > 0]
            if len(vals_pos) == 0:
                expr_pos_means.append(0.0)
            else:
                expr_pos_means.append(float(np.mean(vals_pos)))

        grp["mean_expr_pos"] = expr_pos_means
        grp["gene"] = gene
        grp[STATE_COL] = state_value

        results.append(grp)

    return pd.concat(results, ignore_index=True)

def make_dotplot(
    df_plot,
    genes_order,
    groups_order,
    state_value,
    group_col,
    color_col,
    color_label,
    outpath,
    title_prefix=""
):
    if df_plot.empty:
        print(f"Skipping empty plot: {outpath}")
        return

    plot_df = df_plot.copy()
    plot_df = plot_df[
        plot_df["gene"].isin(genes_order) &
        plot_df[group_col].astype(str).isin([str(g) for g in groups_order])
    ].copy()

    if plot_df.empty:
        print(f"Skipping empty filtered plot: {outpath}")
        return

    gene_to_x = {g: i for i, g in enumerate(genes_order)}
    group_to_y = {g: i for i, g in enumerate(groups_order)}

    plot_df["x"] = plot_df["gene"].map(gene_to_x)
    plot_df["y"] = plot_df[group_col].astype(str).map({str(k): v for k, v in group_to_y.items()})
    plot_df["dot_size"] = pct_to_size(plot_df["pct_expr"].values)

    color_vals = plot_df[color_col].astype(float).values
    finite_vals = color_vals[np.isfinite(color_vals)]

    if len(finite_vals) == 0:
        vmin, vmax = 0.0, 1.0
    else:
        vmin = 0.0
        vmax = np.percentile(finite_vals, 99)
        if not np.isfinite(vmax) or vmax <= 0:
            vmax = float(np.max(finite_vals))
        if not np.isfinite(vmax) or vmax <= 0:
            vmax = 1.0

    n_genes = len(genes_order)
    n_groups = len(groups_order)

    fig_w = max(12, 0.34 * n_genes + 3.4)
    fig_h = max(3.8, 0.62 * n_groups + 1.8)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    sca = ax.scatter(
        plot_df["x"],
        plot_df["y"],
        s=plot_df["dot_size"],
        c=plot_df[color_col],
        cmap=CMAP,
        vmin=vmin,
        vmax=vmax,
        edgecolors="black",
        linewidths=0.5
    )

    ax.set_xticks(range(n_genes))
    ax.set_xticklabels(genes_order, rotation=90, ha="center", fontsize=8)

    ax.set_yticks(range(n_groups))
    ax.set_yticklabels(groups_order, fontsize=9)

    ax.set_xlim(-0.6, n_genes - 0.4)
    ax.set_ylim(-0.6, n_groups - 0.4)
    ax.invert_yaxis()

    ax.set_xlabel("Gene")
    ax.set_ylabel(group_col)

    title = (
        f"{title_prefix}{state_value} | grouped by {group_col}\n"
        f"Dot size = % expressing, color = {color_label}"
    )
    ax.set_title(title)

    ax.set_axisbelow(True)
    ax.grid(axis="x", linestyle=":", alpha=0.25)
    ax.grid(axis="y", linestyle=":", alpha=0.25)

    cbar = fig.colorbar(sca, ax=ax, pad=0.02)
    cbar.set_label(color_label)

    legend_pcts = [1, 5, 10, 25, 50]
    handles = [
        ax.scatter([], [], s=pct_to_size([p])[0], c="lightgray",
                   edgecolors="black", linewidths=0.5)
        for p in legend_pcts
    ]
    labels = [f"{p}%" for p in legend_pcts]
    ax.legend(
        handles, labels,
        title="% expressing",
        bbox_to_anchor=(1.14, 1.0),
        loc="upper left",
        frameon=False
    )

    plt.tight_layout()
    fig.savefig(outpath, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)

# LOAD OBJECT
adata = sc.read_h5ad(ADATA_PATH)

if RESTRICT_LEIDEN:
    adata = adata[adata.obs[LEIDEN_COL].astype(str) == str(LEIDEN_TARGET)].copy()

if COUNTS_LAYER not in adata.layers:
    raise KeyError(
        f"adata.layers['{COUNTS_LAYER}'] not found. "
        f"Available layers: {list(adata.layers.keys())}"
    )

for col in [STATE_COL, COND_COL, COND_CONSOL]:
    if col not in adata.obs.columns:
        raise KeyError(f"Missing required obs column: {col}")

adata = adata[adata.obs[STATE_COL].astype(str).isin(keep_states)].copy()

base_obs = adata.obs[[COND_CONSOL, COND_COL, RUN_COL, STATE_COL]].copy()

# remove WT from DAM using condition_consol logic
drop_mask = (
    (base_obs[STATE_COL].astype(str) == "DAM") &
    (base_obs[COND_CONSOL].astype(str).isin(wt_conds))
)
base_obs = base_obs.loc[~drop_mask].copy()
adata = adata[base_obs.index].copy()

print("adata shape after filtering:", adata.shape)
print("\nCounts after removing WT from DAM:")
print(base_obs.groupby([STATE_COL, COND_CONSOL]).size())

# GENE PRESENCE CHECK
genes_all = adata.var_names.to_list()
marker_present = [g for g in MARKER_GENES if g in genes_all]
marker_missing = [g for g in MARKER_GENES if g not in genes_all]

marker_present_up = [g for g in MARKERS_UP if g in genes_all]
marker_present_down = [g for g in MARKERS_DOWN if g in genes_all]

marker_report = os.path.join(OUT_DIR, "marker_gene_presence_report.txt")
with open(marker_report, "w") as f:
    f.write("Marker genes requested:\n")
    f.write("\n".join(MARKER_GENES))
    f.write("\n\nPresent in adata:\n")
    f.write("\n".join(marker_present))
    f.write("\n\nMissing from adata:\n")
    f.write("\n".join(marker_missing))

print(f"\nMarker genes present: {len(marker_present)} / {len(MARKER_GENES)}")
if marker_missing:
    print("Missing markers:", marker_missing)

# STATE-SPECIFIC ORDERS
present_consol_by_state = {
    state: [c for c in cond_consol_order
            if c in set(base_obs.loc[
                base_obs[STATE_COL].astype(str) == state, COND_CONSOL
            ].astype(str))]
    for state in state_order
}

for state in state_order:
    extras = sorted(
        set(base_obs.loc[
            base_obs[STATE_COL].astype(str) == state, COND_CONSOL
        ].astype(str)) - set(present_consol_by_state[state])
    )
    present_consol_by_state[state] += extras

present_condition_by_state = {
    state: sorted(
        base_obs.loc[
            base_obs[STATE_COL].astype(str) == state, COND_COL
        ].astype(str).unique().tolist(),
        key=family_sort_key
    )
    for state in state_order
}

print("\npresent_consol_by_state:")
for k, v in present_consol_by_state.items():
    print(k, v)

print("\npresent_condition_by_state:")
for k, v in present_condition_by_state.items():
    print(k, v)

# SUMMARIZE
all_summaries = []

for group_col in [COND_CONSOL, COND_COL]:
    for state_value in state_order:
        df_sum = summarize_gene_group(
            df_base=base_obs,
            adata_obj=adata,
            genes=marker_present,
            group_col=group_col,
            state_value=state_value
        )
        all_summaries.append(df_sum)

summary_df = pd.concat(all_summaries, ignore_index=True)
summary_csv = os.path.join(OUT_DIR, "marker_dotplot_summary_long.csv")
summary_df.to_csv(summary_csv, index=False)

# PLOT SPECS
plot_specs = [
    # condition_consol
    (COND_CONSOL, "DAM", marker_present_up,   "UP",   present_consol_by_state["DAM"]),
    (COND_CONSOL, "DAM", marker_present_down, "DOWN", present_consol_by_state["DAM"]),
    (COND_CONSOL, "Homeostatic microglia", marker_present_up,   "UP",   present_consol_by_state["Homeostatic microglia"]),
    (COND_CONSOL, "Homeostatic microglia", marker_present_down, "DOWN", present_consol_by_state["Homeostatic microglia"]),

    # condition
    (COND_COL, "DAM", marker_present_up,   "UP",   present_condition_by_state["DAM"]),
    (COND_COL, "DAM", marker_present_down, "DOWN", present_condition_by_state["DAM"]),
    (COND_COL, "Homeostatic microglia", marker_present_up,   "UP",   present_condition_by_state["Homeostatic microglia"]),
    (COND_COL, "Homeostatic microglia", marker_present_down, "DOWN", present_condition_by_state["Homeostatic microglia"]),
]

# MAKE BOTH COLOR MODES
color_modes = [
    ("mean_expr_all", "Mean counts per cell (all cells)", "populationMean"),
    ("mean_expr_pos", "Mean counts per expressing cell", "expressingOnlyMean"),
]

for color_col, color_label, color_tag in color_modes:
    for group_col, state_value, genes_subset, gene_tag, group_order in plot_specs:
        if len(genes_subset) == 0 or len(group_order) == 0:
            print(f"Skipping empty spec: {group_col}, {state_value}, {gene_tag}, {color_tag}")
            continue

        df_plot = summary_df[
            (summary_df[STATE_COL].astype(str) == state_value) &
            (summary_df["gene"].isin(genes_subset)) &
            (summary_df[group_col].astype(str).isin([str(x) for x in group_order]))
        ].copy()

        group_name = "conditionConsol" if group_col == COND_CONSOL else "condition"
        state_name = "DAM" if state_value == "DAM" else "HOME"

        outpath = os.path.join(
            OUT_DIR,
            f"dotplot_{group_name}_{state_name}_{gene_tag}_{color_tag}.pdf"
        )

        make_dotplot(
            df_plot=df_plot,
            genes_order=genes_subset,
            groups_order=group_order,
            state_value=state_value,
            group_col=group_col,
            color_col=color_col,
            color_label=color_label,
            outpath=outpath
        )
        print("Wrote:", outpath)

print("\nWrote summary table:", summary_csv)
print("Wrote marker report:", marker_report)
print("Done.")

adata shape after filtering: (48182, 347)

Counts after removing WT from DAM:
miranda_cell_type_refined_combinedUpdate  condition_consol
Homeostatic microglia                     WT-Veh               7927
                                          WT-Abe               6625
                                          FAD-Veh             10704
                                          FAD-Abe              8845
DAM                                       WT-Veh                  0
                                          WT-Abe                  0
                                          FAD-Veh              7447
                                          FAD-Abe              6634
dtype: int64

Marker genes present: 67 / 67

present_consol_by_state:
DAM ['FAD-Abe', 'FAD-Veh']
Homeostatic microglia ['FAD-Abe', 'FAD-Veh', 'WT-Abe', 'WT-Veh']

present_condition_by_state:
DAM ['FAD-Abe1', 'FAD-Abe2', 'FAD-Abe3', 'FAD-Veh1', 'FAD-Veh2', 'FAD-Veh3']
Homeostatic microglia ['FAD-Abe1', 'FAD-Abe2', 'FAD

/tmp/ipykernel_3780948/2207602246.py:340: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(base_obs.groupby([STATE_COL, COND_CONSOL]).size())


Wrote: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/marker_dotplots_condition_and_consol_UPDOWN_twoColorModes/dotplot_conditionConsol_DAM_UP_populationMean.pdf
Wrote: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/marker_dotplots_condition_and_consol_UPDOWN_twoColorModes/dotplot_conditionConsol_DAM_DOWN_populationMean.pdf
Wrote: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/marker_dotplots_condition_and_consol_UPDOWN_twoColorModes/dotplot_conditionConsol_HOME_UP_populationMean.pdf
Wrote: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/marker_dotplots_condition_and_consol_UPDOWN_twoColorModes/dotplot_conditionConsol_HOME_DOWN_populationMean.pdf
Wrote: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/marker_dotplots_condition_and_consol_UPDOWN_twoColorModes/dotplot_condition_DAM_UP_populationMean.pdf
Wrote: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/

In [36]:
print(
    summary_df[
        (summary_df["gene"] == "Apoe") &
        (summary_df[STATE_COL] == "DAM")
    ][["condition_consol", "mean_expr_all", "mean_expr_pos", "pct_expr"]]
)

    condition_consol  mean_expr_all  mean_expr_pos   pct_expr
8            FAD-Veh      22.800859      23.291906  97.891768
9            FAD-Abe      23.738167      24.105158  98.477540
426              NaN      30.521557      30.854116  98.922156
427              NaN      22.708275      22.974104  98.842917
428              NaN      19.765152      20.244423  97.632576
429              NaN      31.392107      31.853416  98.551774
430              NaN      19.150047      19.600636  97.701149
431              NaN      14.630969      15.073085  97.066849


In [2]:
# violin 2?

In [44]:
# ============================================================
# MARKER-ONLY VIOLIN PLOTS
# - by replicate-level condition
# - by consolidated condition_consol
#
# % expressing = percent of all cells in the group with raw counts > 0
# ============================================================

import os
import re
import colorsys
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Patch

# ============================================================
# PATHS / SETTINGS
# ============================================================
ADATA_PATH = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/combined/"
    "sub_adata_microglia_leiden6_noOutside_reclustered_gatex2_with_miranda_"
    "DAMvHOME_damifnhome_combined_withCounts_combinedcleaned_UPDATE_palette2.h5ad"
)

OUT_DIR = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/"
    "final_plots/microglia_subset"
)
os.makedirs(OUT_DIR, exist_ok=True)

# replicate-level outputs
OUT_PDF_RAW_CONDITION = os.path.join(
    OUT_DIR,
    "MARKERgenes_allCells_violin_byCondition_pctExpr_ALLDOTS_RAW.pdf"
)
OUT_PDF_LOG1P_CONDITION = os.path.join(
    OUT_DIR,
    "MARKERgenes_allCells_violin_byCondition_pctExpr_ALLDOTS_LOG1P.pdf"
)
OUT_GROUP_CSV_CONDITION = os.path.join(
    OUT_DIR,
    "MARKERgenes_groupSummary_byCondition_long_pctExpr.csv"
)

# consolidated outputs
OUT_PDF_RAW_CONSOL = os.path.join(
    OUT_DIR,
    "MARKERgenes_allCells_violin_byConditionConsol_pctExpr_ALLDOTS_RAW.pdf"
)
OUT_PDF_LOG1P_CONSOL = os.path.join(
    OUT_DIR,
    "MARKERgenes_allCells_violin_byConditionConsol_pctExpr_ALLDOTS_LOG1P.pdf"
)
OUT_GROUP_CSV_CONSOL = os.path.join(
    OUT_DIR,
    "MARKERgenes_groupSummary_byConditionConsol_long_pctExpr.csv"
)

OUT_MARKER_STATUS_TXT = os.path.join(
    OUT_DIR,
    "marker_gene_presence_report.txt"
)

OUT_NAN_REPORT_TXT = os.path.join(
    OUT_DIR,
    "nan_report.txt"
)

STATE_COL = "miranda_cell_type_refined_combinedUpdate"
COND_COL = "condition"
COND_CONSOL = "condition_consol"
RUN_COL = "run_key"
LEIDEN_COL = "leiden"
COUNTS_LAYER = "counts"

RESTRICT_LEIDEN = False
LEIDEN_TARGET = "6"

state_order = ["DAM", "Homeostatic microglia"]
keep_states = set(state_order)

# keep prior DAM exclusion logic
wt_conds = {"WT-Veh", "WT-Abe"}

# base colors by consolidated family
cond_consol_palette = {
    "FAD-Abe": "#1f77b4",
    "FAD-Veh": "#ff7f0e",
    "WT-Abe":  "#2ca02c",
    "WT-Veh":  "#d62728",
}

# violin settings
VIOLIN_CUT = 2
POINT_SIZE = 0.55
POINT_ALPHA = 0.20
POINT_JITTER = 0.18

# annotation settings
PCT_EXPR_TEXT_SIZE = 7
PCT_EXPR_OFFSET_FRAC = 0.025

# axis baseline settings
RAW_YMIN = -1.0
LOG1P_YMIN = -0.5

# ============================================================
# MARKER LISTS
# ============================================================
MARKERS_UP = [
    "Acsbg1", "Gfap", "Axl", "Rmst", "Apoe", "H2-DMa", "Cntn6", "Fcgr1",
    "Cst7", "Cux2", "Fign", "Pdcd1", "Aqp4", "Dpy19l1", "Ccr5",
    "Trem2", "Slc11a1", "Pag1", "Olfml3"
]

MARKERS_DOWN = [
    "Arc", "Sall1", "Mertk", "Cd9", "Cd52", "Foxp1", "Inpp4b", "Hpcal1",
    "Jun", "Nrn1", "Egr1", "2010300C02Rik", "Lpl", "Nrp2", "Serpine2",
    "Gpnmb", "Lgals3", "Wfs1", "Tgfbr1", "Gas7", "Meis2", "Calb1",
    "Havcr2", "Mafb", "Rasgrf2", "Spi1", "Cxcr4", "Tyrobp", "Fabp5",
    "Ctsb", "Tmem163", "Lrpap1", "Crybb1", "Laptm5", "Siglech", "Hexb",
    "Mef2a", "Kctd12", "Rhob", "P2ry12", "Ikzf1", "Ctsd", "Apbb2",
    "Ctsz", "Cd68", "Lat2", "Cx3cr1", "Runx1"
]

MARKER_GENES = list(dict.fromkeys(MARKERS_UP + MARKERS_DOWN))

# ============================================================
# HELPERS
# ============================================================
def get_gene_counts(adata_obj, gene, layer=COUNTS_LAYER):
    X = adata_obj[:, gene].layers[layer]
    if sp.issparse(X):
        return np.asarray(X.toarray()).ravel()
    return np.asarray(X).ravel()

def lighten_hex(hex_color, amount=0.25):
    hex_color = hex_color.lstrip("#")
    r = int(hex_color[0:2], 16) / 255.0
    g = int(hex_color[2:4], 16) / 255.0
    b = int(hex_color[4:6], 16) / 255.0

    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = min(1.0, l + amount * (1 - l))
    r2, g2, b2 = colorsys.hls_to_rgb(h, l, s)

    return "#{:02x}{:02x}{:02x}".format(
        int(round(r2 * 255)),
        int(round(g2 * 255)),
        int(round(b2 * 255)),
    )

def build_condition_palette(groups):
    family_to_base = {
        "FAD-Abe": cond_consol_palette["FAD-Abe"],
        "FAD-Veh": cond_consol_palette["FAD-Veh"],
        "WT-Abe": cond_consol_palette["WT-Abe"],
        "WT-Veh": cond_consol_palette["WT-Veh"],
    }

    fam_to_members = {}
    for g in groups:
        fam = re.sub(r"\d+$", "", str(g))
        fam_to_members.setdefault(fam, []).append(g)

    palette = {}
    for fam, members in fam_to_members.items():
        members = sorted(members)
        base = family_to_base.get(fam, "#808080")

        if len(members) == 1:
            palette[members[0]] = base
        else:
            amounts = np.linspace(0.00, 0.35, len(members))
            for member, amt in zip(members, amounts):
                palette[member] = lighten_hex(base, amount=float(amt))

    return palette

def build_consol_palette(groups):
    return {g: cond_consol_palette.get(g, "#808080") for g in groups}

def annotate_pct_expressing(ax, df_group, mean_col, state_order, hue_order, hue_col):
    hue_offsets_map = {
        1: [0.00],
        2: [-0.20, 0.20],
        3: [-0.25, 0.00, 0.25],
        4: [-0.30, -0.10, 0.10, 0.30],
        5: [-0.34, -0.17, 0.00, 0.17, 0.34],
        6: [-0.36, -0.22, -0.08, 0.08, 0.22, 0.36],
        7: [-0.38, -0.25, -0.13, 0.00, 0.13, 0.25, 0.38],
        8: [-0.40, -0.29, -0.17, -0.06, 0.06, 0.17, 0.29, 0.40],
        9: [-0.42, -0.31, -0.21, -0.10, 0.00, 0.10, 0.21, 0.31, 0.42],
        10: [-0.44, -0.34, -0.24, -0.15, -0.05, 0.05, 0.15, 0.24, 0.34, 0.44],
        11: [-0.45, -0.36, -0.27, -0.18, -0.09, 0.00, 0.09, 0.18, 0.27, 0.36, 0.45],
        12: [-0.46, -0.38, -0.29, -0.21, -0.13, -0.04, 0.04, 0.13, 0.21, 0.29, 0.38, 0.46],
    }

    if len(hue_order) in hue_offsets_map:
        offsets = hue_offsets_map[len(hue_order)]
    else:
        offsets = np.linspace(-0.46, 0.46, len(hue_order))

    state_to_x = {state: i for i, state in enumerate(state_order)}
    hue_to_offset = {h: offsets[i] for i, h in enumerate(hue_order)}

    ylim = ax.get_ylim()
    y_top = ylim[1]
    y_pad = max((y_top - ylim[0]) * PCT_EXPR_OFFSET_FRAC, 0.03)

    for _, row in df_group.iterrows():
        state = str(row[STATE_COL])
        hue = str(row[hue_col])

        if state not in state_to_x or hue not in hue_to_offset:
            continue

        x = state_to_x[state] + hue_to_offset[hue]
        y = float(row[mean_col]) + y_pad
        pct = float(row["pct_expr"])
        y = min(y, y_top * 0.98)

        ax.text(
            x, y, f"{pct:.1f}%",
            ha="center", va="bottom",
            fontsize=PCT_EXPR_TEXT_SIZE,
            color="black"
        )

def make_violin_page(
    df_plot,
    df_group,
    gene,
    y_col,
    mean_col,
    ylabel,
    title_suffix,
    pdf_handle,
    hue_col,
    hue_order,
    palette,
    y_min,
    legend_title
):
    plt.figure(figsize=(max(11.5, 1.25 * len(hue_order) + 6.5), 6.0))
    ax = plt.gca()

    sns.violinplot(
        data=df_plot,
        x=STATE_COL,
        y=y_col,
        hue=hue_col,
        order=state_order,
        hue_order=hue_order,
        palette=palette,
        cut=VIOLIN_CUT,
        density_norm="width",
        inner=None,
        linewidth=1,
        ax=ax
    )

    sns.stripplot(
        data=df_plot,
        x=STATE_COL,
        y=y_col,
        hue=hue_col,
        order=state_order,
        hue_order=hue_order,
        dodge=True,
        jitter=POINT_JITTER,
        size=POINT_SIZE,
        alpha=POINT_ALPHA,
        linewidth=0,
        palette={k: "black" for k in hue_order},
        ax=ax
    )

    sns.stripplot(
        data=df_group,
        x=STATE_COL,
        y=mean_col,
        hue=hue_col,
        order=state_order,
        hue_order=hue_order,
        dodge=True,
        marker="D",
        size=7,
        edgecolor="black",
        linewidth=0.8,
        palette=palette,
        ax=ax
    )

    vals = df_plot[y_col].values
    finite_vals = vals[np.isfinite(vals)]

    if len(finite_vals) == 0:
        ymax = 1.0
    else:
        ymax = np.percentile(finite_vals, 99)
        if not np.isfinite(ymax) or ymax <= 0:
            ymax = float(np.nanmax(finite_vals))
        if not np.isfinite(ymax) or ymax <= 0:
            ymax = 1.0

    y_top = ymax * 1.10
    if y_top <= y_min:
        y_top = y_min + 1.0

    ax.set_ylim(y_min, y_top)

    annotate_pct_expressing(
        ax=ax,
        df_group=df_group,
        mean_col=mean_col,
        state_order=state_order,
        hue_order=hue_order,
        hue_col=hue_col
    )

    ax.axhline(0, color="gray", linestyle="--", linewidth=1, alpha=0.8)

    ax.set_title(f"{gene}\n{title_suffix}")
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)

    if ax.legend_ is not None:
        ax.legend_.remove()

    legend_handles = [
        Patch(facecolor=palette[h], edgecolor="black", label=h)
        for h in hue_order
    ]
    ax.legend(
        handles=legend_handles,
        title=legend_title,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False
    )

    plt.tight_layout()
    pdf_handle.savefig(ax.figure, dpi=200)
    plt.close()

# ============================================================
# LOAD OBJECT
# ============================================================
adata = sc.read_h5ad(ADATA_PATH)

if RESTRICT_LEIDEN:
    adata = adata[adata.obs[LEIDEN_COL].astype(str) == str(LEIDEN_TARGET)].copy()

if COUNTS_LAYER not in adata.layers:
    raise KeyError(
        f"adata.layers['{COUNTS_LAYER}'] not found. "
        f"Available layers: {list(adata.layers.keys())}"
    )

for col in [STATE_COL, COND_COL, COND_CONSOL, RUN_COL]:
    if col not in adata.obs.columns:
        raise KeyError(f"Missing required obs column: {col}")

# ============================================================
# INITIAL GENE CHECK
# ============================================================
genes_all = adata.var_names.to_list()
marker_present = [g for g in MARKER_GENES if g in genes_all]
marker_missing = [g for g in MARKER_GENES if g not in genes_all]

with open(OUT_MARKER_STATUS_TXT, "w") as f:
    f.write("Marker genes requested:\n")
    f.write("\n".join(MARKER_GENES))
    f.write("\n\nPresent in adata:\n")
    f.write("\n".join(marker_present))
    f.write("\n\nMissing from adata:\n")
    f.write("\n".join(marker_missing))

print(f"\nMarker genes present: {len(marker_present)} / {len(MARKER_GENES)}")
if marker_missing:
    print("Missing markers:", marker_missing)

# ============================================================
# NaN REPORT BEFORE FILTERING
# ============================================================
nan_lines = []
nan_lines.append("=== OBS COLUMN NaN COUNTS (before filtering) ===")
for col in [STATE_COL, COND_COL, COND_CONSOL, RUN_COL]:
    n_nan = int(adata.obs[col].isna().sum())
    nan_lines.append(f"{col}: {n_nan}")

nan_lines.append("\n=== COUNTS NaN CHECK (marker genes only) ===")
if len(marker_present) > 0:
    X_markers = adata[:, marker_present].layers[COUNTS_LAYER]
    if sp.issparse(X_markers):
        data_nan = int(np.isnan(X_markers.data).sum())
        nan_lines.append(f"sparse counts .data NaNs: {data_nan}")
        nan_lines.append(f"marker matrix shape: {X_markers.shape} (implicit zeros not NaN)")
    else:
        data_nan = int(np.isnan(X_markers).sum())
        nan_lines.append(f"dense counts NaNs: {data_nan}")
        nan_lines.append(f"marker matrix shape: {X_markers.shape}")
else:
    nan_lines.append("No marker genes present, skipping counts NaN check.")

# ============================================================
# FILTER STATES
# ============================================================
adata = adata[adata.obs[STATE_COL].astype(str).isin(keep_states)].copy()
adata.obs[STATE_COL] = (
    adata.obs[STATE_COL]
    .astype(str)
    .astype("category")
    .cat.remove_unused_categories()
)

# ============================================================
# BASE OBS TABLE + DROP NaNs
# ============================================================
base_obs = adata.obs[[COND_CONSOL, COND_COL, RUN_COL, STATE_COL]].copy()

n_before_dropna = base_obs.shape[0]
base_obs = base_obs.dropna(subset=[COND_CONSOL, COND_COL, RUN_COL, STATE_COL]).copy()
n_after_dropna = base_obs.shape[0]

nan_lines.append("\n=== ROWS REMOVED DUE TO NaNs IN GROUPING COLUMNS ===")
nan_lines.append(f"rows before dropna: {n_before_dropna}")
nan_lines.append(f"rows after dropna:  {n_after_dropna}")
nan_lines.append(f"rows removed:       {n_before_dropna - n_after_dropna}")

adata = adata[base_obs.index].copy()

# ============================================================
# REMOVE WT FROM DAM
# ============================================================
drop_mask = (
    (base_obs[STATE_COL].astype(str) == "DAM") &
    (base_obs[COND_CONSOL].astype(str).isin(wt_conds))
)
base_obs = base_obs.loc[~drop_mask].copy()
adata = adata[base_obs.index].copy()

nan_lines.append("\n=== FINAL SHAPE AFTER ALL CELL FILTERING ===")
nan_lines.append(f"adata shape: {adata.shape}")

with open(OUT_NAN_REPORT_TXT, "w") as f:
    f.write("\n".join(nan_lines))

print("adata shape after filtering:", adata.shape)
print("State counts:")
print(adata.obs[STATE_COL].value_counts())
print("Condition_consol counts:")
print(base_obs[COND_CONSOL].value_counts(dropna=False))
print("Condition counts:")
print(base_obs[COND_COL].value_counts(dropna=False))
print(f"NaN report written to: {OUT_NAN_REPORT_TXT}")

# ============================================================
# ORDERS / PALETTES
# ============================================================
present_conditions = sorted(base_obs[COND_COL].astype(str).unique().tolist())
present_condition_consol = sorted(base_obs[COND_CONSOL].astype(str).unique().tolist())

plot_palette_condition = build_condition_palette(present_conditions)
plot_palette_consol = build_consol_palette(present_condition_consol)

print("\npresent_conditions:", present_conditions)
print("present_condition_consol:", present_condition_consol)

# ============================================================
# LOOP MARKER GENES ONLY
# ============================================================
sns.set_style("whitegrid")

all_group_summaries_condition = []
all_group_summaries_consol = []

with (
    PdfPages(OUT_PDF_RAW_CONDITION) as pdf_raw_condition,
    PdfPages(OUT_PDF_LOG1P_CONDITION) as pdf_log1p_condition,
    PdfPages(OUT_PDF_RAW_CONSOL) as pdf_raw_consol,
    PdfPages(OUT_PDF_LOG1P_CONSOL) as pdf_log1p_consol
):
    for i, gene in enumerate(marker_present, start=1):
        expr_full = get_gene_counts(adata, gene, layer=COUNTS_LAYER)

        df = base_obs.copy()
        idx = adata.obs_names.get_indexer(df.index)
        if np.any(idx < 0):
            raise ValueError(f"Index alignment failed for gene {gene}")

        df["expr"] = expr_full[idx]
        df["expr_log1p"] = np.log1p(df["expr"])
        df["expr_binary"] = (df["expr"] > 0).astype(int)
        df["gene"] = gene

        # ---- replicate-level summary ----
        df_group_condition = (
            df.groupby([COND_COL, STATE_COL], observed=True)
              .agg(
                  mean_expr=("expr", "mean"),
                  median_expr=("expr", "median"),
                  mean_expr_log1p=("expr_log1p", "mean"),
                  median_expr_log1p=("expr_log1p", "median"),
                  n_cells=("expr", "size"),
                  n_expr=("expr_binary", "sum"),
                  pct_expr=("expr_binary", lambda x: 100.0 * np.mean(x)),
              )
              .reset_index()
        )
        df_group_condition["gene"] = gene
        all_group_summaries_condition.append(df_group_condition)

        # ---- consolidated summary ----
        df_group_consol = (
            df.groupby([COND_CONSOL, STATE_COL], observed=True)
              .agg(
                  mean_expr=("expr", "mean"),
                  median_expr=("expr", "median"),
                  mean_expr_log1p=("expr_log1p", "mean"),
                  median_expr_log1p=("expr_log1p", "median"),
                  n_cells=("expr", "size"),
                  n_expr=("expr_binary", "sum"),
                  pct_expr=("expr_binary", lambda x: 100.0 * np.mean(x)),
              )
              .reset_index()
        )
        df_group_consol["gene"] = gene
        all_group_summaries_consol.append(df_group_consol)

        title_condition = (
            "% expressing = % of all cells in each group with raw counts > 0\n"
            "violin=all cells; black dots=all cells; diamonds=mean across cells\n"
            "grouped by replicate-level condition"
        )

        title_consol = (
            "% expressing = % of all cells in each group with raw counts > 0\n"
            "violin=all cells; black dots=all cells; diamonds=mean across cells\n"
            "grouped by consolidated condition"
        )

        # replicate-level raw
        make_violin_page(
            df_plot=df,
            df_group=df_group_condition,
            gene=gene,
            y_col="expr",
            mean_col="mean_expr",
            ylabel="Raw counts per cell",
            title_suffix=title_condition,
            pdf_handle=pdf_raw_condition,
            hue_col=COND_COL,
            hue_order=present_conditions,
            palette=plot_palette_condition,
            y_min=RAW_YMIN,
            legend_title="Condition (replicate-level)"
        )

        # replicate-level log1p
        make_violin_page(
            df_plot=df,
            df_group=df_group_condition,
            gene=gene,
            y_col="expr_log1p",
            mean_col="mean_expr_log1p",
            ylabel="log1p(counts per cell)",
            title_suffix=title_condition,
            pdf_handle=pdf_log1p_condition,
            hue_col=COND_COL,
            hue_order=present_conditions,
            palette=plot_palette_condition,
            y_min=LOG1P_YMIN,
            legend_title="Condition (replicate-level)"
        )

        # consolidated raw
        make_violin_page(
            df_plot=df,
            df_group=df_group_consol,
            gene=gene,
            y_col="expr",
            mean_col="mean_expr",
            ylabel="Raw counts per cell",
            title_suffix=title_consol,
            pdf_handle=pdf_raw_consol,
            hue_col=COND_CONSOL,
            hue_order=present_condition_consol,
            palette=plot_palette_consol,
            y_min=RAW_YMIN,
            legend_title="Condition (consolidated)"
        )

        # consolidated log1p
        make_violin_page(
            df_plot=df,
            df_group=df_group_consol,
            gene=gene,
            y_col="expr_log1p",
            mean_col="mean_expr_log1p",
            ylabel="log1p(counts per cell)",
            title_suffix=title_consol,
            pdf_handle=pdf_log1p_consol,
            hue_col=COND_CONSOL,
            hue_order=present_condition_consol,
            palette=plot_palette_consol,
            y_min=LOG1P_YMIN,
            legend_title="Condition (consolidated)"
        )

        if i % 10 == 0:
            print(f"...{i}/{len(marker_present)} marker genes written")

# ============================================================
# EXPORT CSVs
# ============================================================
gene_order_present = [g for g in MARKER_GENES if g in marker_present]

all_group_summary_df_condition = pd.concat(all_group_summaries_condition, ignore_index=True)
all_group_summary_df_condition["gene"] = pd.Categorical(
    all_group_summary_df_condition["gene"],
    categories=gene_order_present,
    ordered=True
)
all_group_summary_df_condition = all_group_summary_df_condition.sort_values(
    ["gene", COND_COL, STATE_COL]
).reset_index(drop=True)
all_group_summary_df_condition.to_csv(OUT_GROUP_CSV_CONDITION, index=False)

all_group_summary_df_consol = pd.concat(all_group_summaries_consol, ignore_index=True)
all_group_summary_df_consol["gene"] = pd.Categorical(
    all_group_summary_df_consol["gene"],
    categories=gene_order_present,
    ordered=True
)
all_group_summary_df_consol = all_group_summary_df_consol.sort_values(
    ["gene", COND_CONSOL, STATE_COL]
).reset_index(drop=True)
all_group_summary_df_consol.to_csv(OUT_GROUP_CSV_CONSOL, index=False)

print("Finished RAW PDF (condition):", OUT_PDF_RAW_CONDITION)
print("Finished LOG1P PDF (condition):", OUT_PDF_LOG1P_CONDITION)
print("Finished RAW PDF (condition_consol):", OUT_PDF_RAW_CONSOL)
print("Finished LOG1P PDF (condition_consol):", OUT_PDF_LOG1P_CONSOL)
print("Wrote condition summary:", OUT_GROUP_CSV_CONDITION)
print("Wrote condition_consol summary:", OUT_GROUP_CSV_CONSOL)
print("Wrote marker presence report:", OUT_MARKER_STATUS_TXT)
print("Wrote NaN report:", OUT_NAN_REPORT_TXT)
print("Done.")


Marker genes present: 67 / 67
adata shape after filtering: (48182, 347)
State counts:
miranda_cell_type_refined_combinedUpdate
Homeostatic microglia    34101
DAM                      14081
Name: count, dtype: int64
Condition_consol counts:
condition_consol
FAD-Veh    18151
FAD-Abe    15479
WT-Veh      7927
WT-Abe      6625
Name: count, dtype: int64
Condition counts:
condition
FAD-Veh2    7782
FAD-Abe2    6495
FAD-Veh1    6318
FAD-Abe3    4519
FAD-Abe1    4465
FAD-Veh3    4051
WT-Veh2     3442
WT-Veh1     2655
WT-Abe1     2322
WT-Abe3     2166
WT-Abe2     2137
WT-Veh3     1830
Name: count, dtype: int64
NaN report written to: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/final_plots/microglia_subset/nan_report.txt

present_conditions: ['FAD-Abe1', 'FAD-Abe2', 'FAD-Abe3', 'FAD-Veh1', 'FAD-Veh2', 'FAD-Veh3', 'WT-Abe1', 'WT-Abe2', 'WT-Abe3', 'WT-Veh1', 'WT-Veh2', 'WT-Veh3']
present_condition_consol: ['FAD-Abe', 'FAD-Veh', 'WT-Abe', 'WT-Veh']
...10/67 marker genes w

In [20]:
# violin edit 03.30.2026
# ============================================================
# MARKER-ONLY VIOLIN PLOTS
# UPDATED 03/30/2026
#
# CHANGES:
# 1. Output to: microglia_subset/update03302026
# 2. No padding below zero; y-axis starts at 0
# 3. Plot per mouse:
#    Mouse 1 plots = FAD-Veh1, FAD-Abe1, WT-Veh1, WT-Abe1
#    Mouse 2 plots = ...
#    Mouse 3 plots = ...
# 4. Remove on-plot % expressing labels
# 5. Replace mean diamond with short horizontal line colored by
#    condition_consol family color
#
# NOTE:
# - This version replaces the prior condition / condition_consol PDF outputs
#   with per-mouse PDFs.
# - pct_expr is still calculated and saved in the CSV.
# ============================================================

import os
import re
import colorsys
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp

from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Patch

# ============================================================
# PATHS / SETTINGS
# ============================================================
ADATA_PATH = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/combined/"
    "sub_adata_microglia_leiden6_noOutside_reclustered_gatex2_with_miranda_"
    "DAMvHOME_damifnhome_combined_withCounts_combinedcleaned_UPDATE_palette2.h5ad"
)

OUT_DIR = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/"
    "final_plots/microglia_subset/update03302026-3"
)
os.makedirs(OUT_DIR, exist_ok=True)

OUT_MARKER_STATUS_TXT = os.path.join(OUT_DIR, "marker_gene_presence_report.txt")
OUT_NAN_REPORT_TXT = os.path.join(OUT_DIR, "nan_report.txt")
OUT_GROUP_CSV = os.path.join(OUT_DIR, "MARKERgenes_groupSummary_byMouseCondition_long.csv")

STATE_COL = "miranda_cell_type_refined_combinedUpdate"
COND_COL = "condition"
COND_CONSOL = "condition_consol"
RUN_COL = "run_key"
LEIDEN_COL = "leiden"
COUNTS_LAYER = "counts"

RESTRICT_LEIDEN = False
LEIDEN_TARGET = "6"

state_order = ["DAM", "Homeostatic microglia"]
keep_states = set(state_order)

# keep prior DAM exclusion logic
wt_conds = {"WT-Veh", "WT-Abe"}

# base colors by consolidated family
cond_consol_palette = {
    "FAD-Abe": "#1f77b4",
    "FAD-Veh": "#ff7f0e",
    "WT-Abe":  "#2ca02c",
    "WT-Veh":  "#d62728",
}

# one-mouse desired condition order
condition_family_order = ["FAD-Abe", "FAD-Veh", "WT-Abe", "WT-Veh"]

# violin settings
VIOLIN_CUT = 2
POINT_SIZE = 0.55
POINT_ALPHA = 0.20
POINT_JITTER = 0.18

# axis baseline settings: start exactly at 0
RAW_YMIN = 0.0
LOG1P_YMIN = 0.0

# short horizontal mean line settings
MEAN_LINE_HALF_WIDTH = 0.07
MEAN_LINE_WIDTH = 3
LIGHTEN_AMOUNT = 0.5
# ============================================================
# MARKER LISTS
# ============================================================
MARKERS_UP = [
    "Acsbg1", "Gfap", "Axl", "Rmst", "Apoe", "H2-DMa", "Cntn6", "Fcgr1",
    "Cst7", "Cux2", "Fign", "Pdcd1", "Aqp4", "Dpy19l1", "Ccr5",
    "Trem2", "Slc11a1", "Pag1", "Olfml3"
]

MARKERS_DOWN = [
    "Arc", "Sall1", "Mertk", "Cd9", "Cd52", "Foxp1", "Inpp4b", "Hpcal1",
    "Jun", "Nrn1", "Egr1", "2010300C02Rik", "Lpl", "Nrp2", "Serpine2",
    "Gpnmb", "Lgals3", "Wfs1", "Tgfbr1", "Gas7", "Meis2", "Calb1",
    "Havcr2", "Mafb", "Rasgrf2", "Spi1", "Cxcr4", "Tyrobp", "Fabp5",
    "Ctsb", "Tmem163", "Lrpap1", "Crybb1", "Laptm5", "Siglech", "Hexb",
    "Mef2a", "Kctd12", "Rhob", "P2ry12", "Ikzf1", "Ctsd", "Apbb2",
    "Ctsz", "Cd68", "Lat2", "Cx3cr1", "Runx1"
]

MARKER_GENES = list(dict.fromkeys(MARKERS_UP + MARKERS_DOWN))

# ============================================================
# HELPERS
# ============================================================
def get_gene_counts(adata_obj, gene, layer=COUNTS_LAYER):
    X = adata_obj[:, gene].layers[layer]
    if sp.issparse(X):
        return np.asarray(X.toarray()).ravel()
    return np.asarray(X).ravel()

def lighten_hex(hex_color, amount=LIGHTEN_AMOUNT):
    hex_color = hex_color.lstrip("#")
    r = int(hex_color[0:2], 16) / 255.0
    g = int(hex_color[2:4], 16) / 255.0
    b = int(hex_color[4:6], 16) / 255.0

    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = min(1.0, l + amount * (1 - l))
    r2, g2, b2 = colorsys.hls_to_rgb(h, l, s)

    return "#{:02x}{:02x}{:02x}".format(
        int(round(r2 * 255)),
        int(round(g2 * 255)),
        int(round(b2 * 255)),
    )

def build_condition_palette(groups):
    family_to_base = {
        "FAD-Abe": cond_consol_palette["FAD-Abe"],
        "FAD-Veh": cond_consol_palette["FAD-Veh"],
        "WT-Abe": cond_consol_palette["WT-Abe"],
        "WT-Veh": cond_consol_palette["WT-Veh"],
    }

    fam_to_members = {}
    for g in groups:
        fam = re.sub(r"\d+$", "", str(g))
        fam_to_members.setdefault(fam, []).append(g)

    palette = {}
    for fam, members in fam_to_members.items():
        members = sorted(members)
        base = family_to_base.get(fam, "#808080")

        if len(members) == 1:
            palette[members[0]] = base
        else:
            amounts = np.linspace(0.00, 0.35, len(members))
            for member, amt in zip(members, amounts):
                palette[member] = lighten_hex(base, amount=float(amt))

    return palette

def extract_mouse_id(condition_value):
    m = re.search(r"(\d+)$", str(condition_value))
    return m.group(1) if m else np.nan

def family_from_condition(condition_value):
    return re.sub(r"\d+$", "", str(condition_value))

def make_mouse_condition_order(mouse_id, available_conditions):
    desired = [f"{fam}{mouse_id}" for fam in condition_family_order]
    return [c for c in desired if c in set(available_conditions)]

def get_dodge_offsets(n_hues):
    hue_offsets_map = {
        1: [0.00],
        2: [-0.20, 0.20],
        3: [-0.25, 0.00, 0.25],
        4: [-0.30, -0.10, 0.10, 0.30],
        5: [-0.34, -0.17, 0.00, 0.17, 0.34],
        6: [-0.36, -0.22, -0.08, 0.08, 0.22, 0.36],
        7: [-0.38, -0.25, -0.13, 0.00, 0.13, 0.25, 0.38],
        8: [-0.40, -0.29, -0.17, -0.06, 0.06, 0.17, 0.29, 0.40],
        9: [-0.42, -0.31, -0.21, -0.10, 0.00, 0.10, 0.21, 0.31, 0.42],
        10: [-0.44, -0.34, -0.24, -0.15, -0.05, 0.05, 0.15, 0.24, 0.34, 0.44],
        11: [-0.45, -0.36, -0.27, -0.18, -0.09, 0.00, 0.09, 0.18, 0.27, 0.36, 0.45],
        12: [-0.46, -0.38, -0.29, -0.21, -0.13, -0.04, 0.04, 0.13, 0.21, 0.29, 0.38, 0.46],
    }
    if n_hues in hue_offsets_map:
        return hue_offsets_map[n_hues]
    return np.linspace(-0.46, 0.46, n_hues).tolist()

def draw_group_mean_lines(ax, df_group, mean_col, hue_col, hue_order, mean_color_map):
    offsets = get_dodge_offsets(len(hue_order))
    state_to_x = {state: i for i, state in enumerate(state_order)}
    hue_to_offset = {h: offsets[i] for i, h in enumerate(hue_order)}

    for _, row in df_group.iterrows():
        state = str(row[STATE_COL])
        hue = str(row[hue_col])

        if state not in state_to_x or hue not in hue_to_offset:
            continue

        x_center = state_to_x[state] + hue_to_offset[hue]
        y = float(row[mean_col])
        line_color = mean_color_map.get(hue, "black")

        ax.hlines(
            y=y,
            xmin=x_center - MEAN_LINE_HALF_WIDTH,
            xmax=x_center + MEAN_LINE_HALF_WIDTH,
            colors=line_color,
            linewidth=MEAN_LINE_WIDTH,
            alpha=1,
            linestyles="solid",
            zorder=7
        )

def make_violin_page(
    df_plot,
    df_group,
    gene,
    y_col,
    mean_col,
    ylabel,
    title_suffix,
    pdf_handle,
    hue_col,
    hue_order,
    violin_palette,
    mean_color_map,
    y_min,
    legend_title
):
    plt.figure(figsize=(max(8.5, 1.1 * len(hue_order) + 5.5), 6.0))
    ax = plt.gca()

    sns.violinplot(
        data=df_plot,
        x=STATE_COL,
        y=y_col,
        hue=hue_col,
        order=state_order,
        hue_order=hue_order,
        palette=violin_palette,
        cut=VIOLIN_CUT,
        density_norm="width",
        inner=None,
        linewidth=1,
        ax=ax
    )

    sns.stripplot(
        data=df_plot,
        x=STATE_COL,
        y=y_col,
        hue=hue_col,
        order=state_order,
        hue_order=hue_order,
        dodge=True,
        jitter=POINT_JITTER,
        size=POINT_SIZE,
        alpha=POINT_ALPHA,
        linewidth=0,
        palette={k: "black" for k in hue_order},
        ax=ax,
        zorder=5
    )

    # remove any auto legend before custom legend
    if ax.legend_ is not None:
        ax.legend_.remove()

    # draw short horizontal mean lines
    draw_group_mean_lines(
        ax=ax,
        df_group=df_group,
        mean_col=mean_col,
        hue_col=hue_col,
        hue_order=hue_order,
        mean_color_map=mean_color_map
    )

    vals = df_plot[y_col].values
    finite_vals = vals[np.isfinite(vals)]

    if len(finite_vals) == 0:
        ymax = 1.0
    else:
        ymax = np.percentile(finite_vals, 99)
        if not np.isfinite(ymax) or ymax <= 0:
            ymax = float(np.nanmax(finite_vals))
        if not np.isfinite(ymax) or ymax <= 0:
            ymax = 1.0

    y_top = ymax * 1.10
    if y_top <= y_min:
        y_top = y_min + 1.0

    ax.set_ylim(y_min, y_top)

    ax.set_title(f"{gene}\n{title_suffix}")
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)

    legend_handles = [
        Patch(facecolor=violin_palette[h], edgecolor="black", label=h)
        for h in hue_order
    ]
    ax.legend(
        handles=legend_handles,
        title=legend_title,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False
    )

    plt.tight_layout()
    pdf_handle.savefig(ax.figure, dpi=200)
    plt.close()

# ============================================================
# LOAD OBJECT
# ============================================================
adata = sc.read_h5ad(ADATA_PATH)

if RESTRICT_LEIDEN:
    adata = adata[adata.obs[LEIDEN_COL].astype(str) == str(LEIDEN_TARGET)].copy()

if COUNTS_LAYER not in adata.layers:
    raise KeyError(
        f"adata.layers['{COUNTS_LAYER}'] not found. "
        f"Available layers: {list(adata.layers.keys())}"
    )

for col in [STATE_COL, COND_COL, COND_CONSOL, RUN_COL]:
    if col not in adata.obs.columns:
        raise KeyError(f"Missing required obs column: {col}")

# ============================================================
# INITIAL GENE CHECK
# ============================================================
genes_all = adata.var_names.to_list()
marker_present = [g for g in MARKER_GENES if g in genes_all]
marker_missing = [g for g in MARKER_GENES if g not in genes_all]

with open(OUT_MARKER_STATUS_TXT, "w") as f:
    f.write("Marker genes requested:\n")
    f.write("\n".join(MARKER_GENES))
    f.write("\n\nPresent in adata:\n")
    f.write("\n".join(marker_present))
    f.write("\n\nMissing from adata:\n")
    f.write("\n".join(marker_missing))

print(f"\nMarker genes present: {len(marker_present)} / {len(MARKER_GENES)}")
if marker_missing:
    print("Missing markers:", marker_missing)

# ============================================================
# NaN REPORT BEFORE FILTERING
# ============================================================
nan_lines = []
nan_lines.append("=== OBS COLUMN NaN COUNTS (before filtering) ===")
for col in [STATE_COL, COND_COL, COND_CONSOL, RUN_COL]:
    n_nan = int(adata.obs[col].isna().sum())
    nan_lines.append(f"{col}: {n_nan}")

nan_lines.append("\n=== COUNTS NaN CHECK (marker genes only) ===")
if len(marker_present) > 0:
    X_markers = adata[:, marker_present].layers[COUNTS_LAYER]
    if sp.issparse(X_markers):
        data_nan = int(np.isnan(X_markers.data).sum())
        nan_lines.append(f"sparse counts .data NaNs: {data_nan}")
        nan_lines.append(f"marker matrix shape: {X_markers.shape} (implicit zeros not NaN)")
    else:
        data_nan = int(np.isnan(X_markers).sum())
        nan_lines.append(f"dense counts NaNs: {data_nan}")
        nan_lines.append(f"marker matrix shape: {X_markers.shape}")
else:
    nan_lines.append("No marker genes present, skipping counts NaN check.")

# ============================================================
# FILTER STATES
# ============================================================
adata = adata[adata.obs[STATE_COL].astype(str).isin(keep_states)].copy()
adata.obs[STATE_COL] = (
    adata.obs[STATE_COL]
    .astype(str)
    .astype("category")
    .cat.remove_unused_categories()
)

# ============================================================
# BASE OBS TABLE + DROP NaNs
# ============================================================
base_obs = adata.obs[[COND_CONSOL, COND_COL, RUN_COL, STATE_COL]].copy()

n_before_dropna = base_obs.shape[0]
base_obs = base_obs.dropna(subset=[COND_CONSOL, COND_COL, RUN_COL, STATE_COL]).copy()
n_after_dropna = base_obs.shape[0]

nan_lines.append("\n=== ROWS REMOVED DUE TO NaNs IN GROUPING COLUMNS ===")
nan_lines.append(f"rows before dropna: {n_before_dropna}")
nan_lines.append(f"rows after dropna:  {n_after_dropna}")
nan_lines.append(f"rows removed:       {n_before_dropna - n_after_dropna}")

adata = adata[base_obs.index].copy()

# ============================================================
# REMOVE WT FROM DAM
# ============================================================
drop_mask = (
    (base_obs[STATE_COL].astype(str) == "DAM") &
    (base_obs[COND_CONSOL].astype(str).isin(wt_conds))
)
base_obs = base_obs.loc[~drop_mask].copy()
adata = adata[base_obs.index].copy()

# add mouse_id after filtering
base_obs["mouse_id"] = base_obs[COND_COL].map(extract_mouse_id)
base_obs["condition_family"] = base_obs[COND_COL].map(family_from_condition)

nan_lines.append("\n=== MOUSE ID NaN COUNTS ===")
nan_lines.append(f"mouse_id: {int(base_obs['mouse_id'].isna().sum())}")

nan_lines.append("\n=== FINAL SHAPE AFTER ALL CELL FILTERING ===")
nan_lines.append(f"adata shape: {adata.shape}")

with open(OUT_NAN_REPORT_TXT, "w") as f:
    f.write("\n".join(nan_lines))

print("adata shape after filtering:", adata.shape)
print("State counts:")
print(adata.obs[STATE_COL].value_counts())
print("Condition counts:")
print(base_obs[COND_COL].value_counts(dropna=False))
print("Mouse IDs:")
print(base_obs["mouse_id"].value_counts(dropna=False).sort_index())
print(f"NaN report written to: {OUT_NAN_REPORT_TXT}")

# ============================================================
# MOUSE IDS / OUTPUTS
# ============================================================
mouse_ids = sorted([m for m in base_obs["mouse_id"].dropna().unique().tolist()], key=lambda x: int(x))

mouse_pdf_paths_raw = {
    mouse_id: os.path.join(
        OUT_DIR,
        f"MARKERgenes_allCells_violin_byMouse{mouse_id}_RAW.pdf"
    )
    for mouse_id in mouse_ids
}

mouse_pdf_paths_log1p = {
    mouse_id: os.path.join(
        OUT_DIR,
        f"MARKERgenes_allCells_violin_byMouse{mouse_id}_LOG1P.pdf"
    )
    for mouse_id in mouse_ids
}

# ============================================================
# LOOP MARKER GENES ONLY
# ============================================================
sns.set_style("whitegrid")

all_group_summaries = []

pdf_contexts = []
try:
    pdf_handles_raw = {m: PdfPages(path) for m, path in mouse_pdf_paths_raw.items()}
    pdf_handles_log1p = {m: PdfPages(path) for m, path in mouse_pdf_paths_log1p.items()}

    for i, gene in enumerate(marker_present, start=1):
        expr_full = get_gene_counts(adata, gene, layer=COUNTS_LAYER)

        df = base_obs.copy()
        idx = adata.obs_names.get_indexer(df.index)
        if np.any(idx < 0):
            raise ValueError(f"Index alignment failed for gene {gene}")

        df["expr"] = expr_full[idx]
        df["expr_log1p"] = np.log1p(df["expr"])
        df["expr_binary"] = (df["expr"] > 0).astype(int)
        df["gene"] = gene

        for mouse_id in mouse_ids:
            df_mouse = df[df["mouse_id"].astype(str) == str(mouse_id)].copy()
            if df_mouse.empty:
                continue
            
            df_mouse[COND_COL] = df_mouse[COND_COL].astype(str)
            df_mouse[COND_CONSOL] = df_mouse[COND_CONSOL].astype(str)
            df_mouse[STATE_COL] = df_mouse[STATE_COL].astype(str)
            
            hue_order_mouse = make_mouse_condition_order(
                mouse_id=mouse_id,
                available_conditions=df_mouse[COND_COL].unique().tolist()
            )
            if len(hue_order_mouse) == 0:
                continue

            violin_palette_mouse = build_condition_palette(hue_order_mouse)
            mean_color_map_mouse = {
                cond: lighten_hex(cond_consol_palette.get(family_from_condition(cond), "#000000"), amount=LIGHTEN_AMOUNT)
                for cond in hue_order_mouse
            }

            df_group_mouse = (
                df_mouse.groupby([COND_COL, COND_CONSOL, "mouse_id", STATE_COL], observed=True)
                        .agg(
                            mean_expr=("expr", "mean"),
                            median_expr=("expr", "median"),
                            mean_expr_log1p=("expr_log1p", "mean"),
                            median_expr_log1p=("expr_log1p", "median"),
                            n_cells=("expr", "size"),
                            n_expr=("expr_binary", "sum"),
                            pct_expr=("expr_binary", lambda x: 100.0 * np.mean(x)),
                        )
                        .reset_index()
            )
            df_group_mouse["gene"] = gene
            all_group_summaries.append(df_group_mouse)

            title_mouse = (
                f"Mouse {mouse_id}\n"
                "violin=all cells; black dots=all cells; colored horizontal line=mean across cells"
            )

            make_violin_page(
                df_plot=df_mouse,
                df_group=df_group_mouse,
                gene=gene,
                y_col="expr",
                mean_col="mean_expr",
                ylabel="Raw counts per cell",
                title_suffix=title_mouse,
                pdf_handle=pdf_handles_raw[mouse_id],
                hue_col=COND_COL,
                hue_order=hue_order_mouse,
                violin_palette=violin_palette_mouse,
                mean_color_map=mean_color_map_mouse,
                y_min=RAW_YMIN,
                legend_title=f"Mouse {mouse_id} conditions"
            )

            make_violin_page(
                df_plot=df_mouse,
                df_group=df_group_mouse,
                gene=gene,
                y_col="expr_log1p",
                mean_col="mean_expr_log1p",
                ylabel="log1p(counts per cell)",
                title_suffix=title_mouse,
                pdf_handle=pdf_handles_log1p[mouse_id],
                hue_col=COND_COL,
                hue_order=hue_order_mouse,
                violin_palette=violin_palette_mouse,
                mean_color_map=mean_color_map_mouse,
                y_min=LOG1P_YMIN,
                legend_title=f"Mouse {mouse_id} conditions"
            )

        if i % 10 == 0:
            print(f"...{i}/{len(marker_present)} marker genes written")

finally:
    for handle in list(locals().get("pdf_handles_raw", {}).values()):
        handle.close()
    for handle in list(locals().get("pdf_handles_log1p", {}).values()):
        handle.close()

# ============================================================
# EXPORT CSV
# ============================================================
gene_order_present = [g for g in MARKER_GENES if g in marker_present]

if len(all_group_summaries) > 0:
    all_group_summary_df = pd.concat(all_group_summaries, ignore_index=True)
    all_group_summary_df["gene"] = pd.Categorical(
        all_group_summary_df["gene"],
        categories=gene_order_present,
        ordered=True
    )
    all_group_summary_df = all_group_summary_df.sort_values(
        ["mouse_id", "gene", COND_COL, STATE_COL]
    ).reset_index(drop=True)
    all_group_summary_df.to_csv(OUT_GROUP_CSV, index=False)

    print("Wrote group summary:", OUT_GROUP_CSV)
else:
    print("No group summaries were generated.")

print("Mouse RAW PDFs:")
for m in mouse_ids:
    print(f"  Mouse {m}: {mouse_pdf_paths_raw[m]}")

print("Mouse LOG1P PDFs:")
for m in mouse_ids:
    print(f"  Mouse {m}: {mouse_pdf_paths_log1p[m]}")

print("Wrote marker presence report:", OUT_MARKER_STATUS_TXT)
print("Wrote NaN report:", OUT_NAN_REPORT_TXT)
print("Done.")


Marker genes present: 67 / 67
adata shape after filtering: (48182, 347)
State counts:
miranda_cell_type_refined_combinedUpdate
Homeostatic microglia    34101
DAM                      14081
Name: count, dtype: int64
Condition counts:
condition
FAD-Veh2    7782
FAD-Abe2    6495
FAD-Veh1    6318
FAD-Abe3    4519
FAD-Abe1    4465
FAD-Veh3    4051
WT-Veh2     3442
WT-Veh1     2655
WT-Abe1     2322
WT-Abe3     2166
WT-Abe2     2137
WT-Veh3     1830
Name: count, dtype: int64
Mouse IDs:
mouse_id
1    15760
2    19856
3    12566
Name: count, dtype: int64
NaN report written to: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/final_plots/microglia_subset/update03302026-3/nan_report.txt
...10/67 marker genes written
...20/67 marker genes written
...30/67 marker genes written
...40/67 marker genes written
...50/67 marker genes written
...60/67 marker genes written
Wrote group summary: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/final_plots/microgli

In [22]:
# ============================================================
# ALL-GENE SINGLE-MOUSE TPC TABLES + POOLED P-VALUES
#
# Goal:
#   For each comparison (e.g. FAD-Veh vs FAD-Abe),
#   generate CSV tables for all genes with:
#     - pooled mean TPC in each condition
#     - pooled Welch p-value across cells
#     - BH-FDR
#     - per-mouse mean TPC columns (mouse 1/2/3)
#
# Notes:
#   - keeps ALL genes in the object (no mean TPC filtering)
#   - uses updated object + updated annotation column directly
#   - same pooled-cell Welch logic as volcano code, but table output
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

# ============================================================
# PATHS
# ============================================================
ADATA_PATH = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/combined/"
    "sub_adata_microglia_leiden6_noOutside_reclustered_gatex2_with_miranda_"
    "DAMvHOME_damifnhome_combined_withCounts_combinedcleaned_UPDATE_palette2.h5ad"
)

OUT_DIR = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/"
    "final_plots/microglia_subset/allGenes_singleMouseTPC_tables_04032026"
)
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
STATE_COL = "miranda_cell_type_refined_combinedUpdate"
COND_COL = "condition_consol"
COND_MOUSE_COL = "condition"
LEIDEN_COL = "leiden"
COUNTS_LAYER = "counts"

RESTRICT_LEIDEN = False
LEIDEN_TARGET = "6"

KEEP_STATES = ["DAM", "Homeostatic microglia"]

# keep same DAM exclusion logic as the violin code
EXCLUDE_WT_FROM_DAM = True
WT_CONDS = {"WT-Veh", "WT-Abe"}

COMPARISONS = [
    ("FAD-Veh", "FAD-Abe"),
    ("FAD-Veh", "WT-Veh"),
    ("WT-Veh", "WT-Abe"),
]

PSEUDOCOUNT = 1e-6
PVALUE_FLOOR = 1e-300

# keep all genes for this task
APPLY_MEAN_TPC_FILTER = False
MEAN_TPC_MIN = 0.075

# split output by state / pooled group, same idea as volcano code
GROUPINGS = {
    "HOME": ["Homeostatic microglia"],
    "DAM": ["DAM"],
    "TOTAL_MICROGLIA": ["Homeostatic microglia", "DAM"],
}

# ============================================================
# HELPERS
# ============================================================
def extract_mouse_id(condition_value):
    m = re.search(r"(\d+)$", str(condition_value))
    return m.group(1) if m else np.nan


def get_counts_matrix(ad, layer_name=COUNTS_LAYER):
    if layer_name in ad.layers:
        return ad.layers[layer_name]
    return ad.X


def get_gene_vector(X, idx, gi):
    if sp.issparse(X):
        return X[idx, gi].toarray().ravel()
    return np.asarray(X[idx, gi]).ravel()


def add_bh_fdr(df, p_col="pval"):
    df = df.copy()
    pvals = df[p_col].to_numpy(dtype=float)

    df["p_adj"] = np.nan
    valid = np.isfinite(pvals)

    if valid.sum() > 0:
        _, padj, _, _ = multipletests(pvals[valid], method="fdr_bh")
        df.loc[df.index[valid], "p_adj"] = padj

    df["pval_clipped"] = df["pval"].clip(lower=PVALUE_FLOOR)
    df["neglog10_p"] = -np.log10(df["pval_clipped"])

    df["p_adj_clipped"] = df["p_adj"].clip(lower=PVALUE_FLOOR)
    df["neglog10_p_adj"] = -np.log10(df["p_adj_clipped"])

    return df


def maybe_apply_mean_tpc_filter(df, cond_a, cond_b, min_tpc=0.075):
    if not APPLY_MEAN_TPC_FILTER:
        return df

    keep = (df[f"mean_{cond_a}"] >= min_tpc) & (df[f"mean_{cond_b}"] >= min_tpc)
    return df.loc[keep].copy()


# ============================================================
# LOAD / FILTER OBJECT
# ============================================================
ad = sc.read_h5ad(ADATA_PATH)

if RESTRICT_LEIDEN:
    ad = ad[ad.obs[LEIDEN_COL].astype(str) == str(LEIDEN_TARGET)].copy()

for col in [STATE_COL, COND_COL, COND_MOUSE_COL]:
    if col not in ad.obs.columns:
        raise KeyError(f"Missing required obs column: {col}")

if COUNTS_LAYER not in ad.layers:
    raise KeyError(
        f"adata.layers['{COUNTS_LAYER}'] not found. "
        f"Available layers: {list(ad.layers.keys())}"
    )

# keep only DAM / Homeostatic microglia
ad = ad[ad.obs[STATE_COL].astype(str).isin(KEEP_STATES)].copy()
ad.obs[STATE_COL] = (
    ad.obs[STATE_COL]
    .astype(str)
    .astype("category")
    .cat.remove_unused_categories()
)

# optional: preserve prior violin logic removing WT from DAM
if EXCLUDE_WT_FROM_DAM:
    drop_mask = (
        (ad.obs[STATE_COL].astype(str) == "DAM") &
        (ad.obs[COND_COL].astype(str).isin(WT_CONDS))
    )
    ad = ad[~drop_mask].copy()

ad.obs["mouse_id"] = ad.obs[COND_MOUSE_COL].map(extract_mouse_id)

# drop rows with missing grouping fields
keep_idx = ad.obs.dropna(
    subset=[COND_COL, COND_MOUSE_COL, STATE_COL, "mouse_id"]
).index
ad = ad[keep_idx].copy()

print("Filtered adata shape:", ad.shape)
print("\nState counts:")
print(ad.obs[STATE_COL].value_counts(dropna=False))
print("\nCondition_consol counts:")
print(ad.obs[COND_COL].value_counts(dropna=False))
print("\nMouse counts:")
print(ad.obs["mouse_id"].value_counts(dropna=False).sort_index())

# ============================================================
# MAIN
# ============================================================
X = get_counts_matrix(ad, COUNTS_LAYER)
genes = ad.var_names.to_list()

cond_vals = ad.obs[COND_COL].astype(str).values
mouse_vals = ad.obs["mouse_id"].astype(str).values
state_vals = ad.obs[STATE_COL].astype(str).values

for cond_a, cond_b in COMPARISONS:
    print(f"\n=== {cond_a} vs {cond_b} ===")

    for group_name, keep_states in GROUPINGS.items():
        idx_group = np.where(np.isin(state_vals, keep_states))[0]
        if idx_group.size == 0:
            print(f"  {group_name}: no cells")
            continue

        idxA = idx_group[cond_vals[idx_group] == str(cond_a)]
        idxB = idx_group[cond_vals[idx_group] == str(cond_b)]

        if idxA.size == 0 or idxB.size == 0:
            print(f"  {group_name}: skipped (missing one side)")
            continue

        mouse_ids_a = sorted(np.unique(mouse_vals[idxA]).tolist(), key=lambda x: int(x))
        mouse_ids_b = sorted(np.unique(mouse_vals[idxB]).tolist(), key=lambda x: int(x))

        rows = []

        for gi, gene in enumerate(genes):
            a_vals = get_gene_vector(X, idxA, gi)
            b_vals = get_gene_vector(X, idxB, gi)

            try:
                stat, p = ttest_ind(a_vals, b_vals, equal_var=False, nan_policy="omit")
            except Exception:
                stat, p = (np.nan, np.nan)

            mean_a = float(np.mean(a_vals))
            mean_b = float(np.mean(b_vals))
            log2fc = np.log2((mean_a + PSEUDOCOUNT) / (mean_b + PSEUDOCOUNT))

            row = {
                "group": group_name,
                "gene": gene,
                "condition_A": cond_a,
                "condition_B": cond_b,
                "states_included": "|".join(keep_states),
                "n_cells_A": int(idxA.size),
                "n_cells_B": int(idxB.size),
                f"mean_{cond_a}": mean_a,
                f"mean_{cond_b}": mean_b,
                "log2FC": log2fc,
                "t_stat": stat,
                "pval": p,
                "state_col_used": STATE_COL,
                "condition_col_used": COND_COL,
            }

            # per-mouse mean TPC columns for condition A
            for m in mouse_ids_a:
                idxA_m = idxA[mouse_vals[idxA] == str(m)]
                vals_m = get_gene_vector(X, idxA_m, gi) if idxA_m.size > 0 else np.array([])
                row[f"{cond_a}_mouse{m}_meanTPC"] = float(np.mean(vals_m)) if vals_m.size > 0 else np.nan
                row[f"{cond_a}_mouse{m}_nCells"] = int(idxA_m.size)

            # per-mouse mean TPC columns for condition B
            for m in mouse_ids_b:
                idxB_m = idxB[mouse_vals[idxB] == str(m)]
                vals_m = get_gene_vector(X, idxB_m, gi) if idxB_m.size > 0 else np.array([])
                row[f"{cond_b}_mouse{m}_meanTPC"] = float(np.mean(vals_m)) if vals_m.size > 0 else np.nan
                row[f"{cond_b}_mouse{m}_nCells"] = int(idxB_m.size)

            rows.append(row)

        res = pd.DataFrame(rows)
        if res.empty:
            print(f"  {group_name}: no results")
            continue

        res = add_bh_fdr(res, p_col="pval")
        res = maybe_apply_mean_tpc_filter(res, cond_a, cond_b, min_tpc=MEAN_TPC_MIN)

        out_csv = os.path.join(
            OUT_DIR,
            f"allGenes_singleMouseTPC_{group_name}_{cond_a}_vs_{cond_b}.csv"
        )
        res.to_csv(out_csv, index=False)
        print(f"  Wrote: {out_csv}")

print("\nDone.")
print("Output folder:", OUT_DIR)

Filtered adata shape: (48182, 347)

State counts:
miranda_cell_type_refined_combinedUpdate
Homeostatic microglia    34101
DAM                      14081
Name: count, dtype: int64

Condition_consol counts:
condition_consol
FAD-Veh    18151
FAD-Abe    15479
WT-Veh      7927
WT-Abe      6625
Name: count, dtype: int64

Mouse counts:
mouse_id
1    15760
2    19856
3    12566
Name: count, dtype: int64

=== FAD-Veh vs FAD-Abe ===
  Wrote: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/final_plots/microglia_subset/allGenes_singleMouseTPC_tables_04032026/allGenes_singleMouseTPC_HOME_FAD-Veh_vs_FAD-Abe.csv
  Wrote: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/final_plots/microglia_subset/allGenes_singleMouseTPC_tables_04032026/allGenes_singleMouseTPC_DAM_FAD-Veh_vs_FAD-Abe.csv
  Wrote: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/final_plots/microglia_subset/allGenes_singleMouseTPC_tables_04032026/allGenes_singleMouseTPC

In [4]:
adata.obs.condition.value_counts()

condition
FAD-Veh2    7782
FAD-Abe2    6495
FAD-Veh1    6318
FAD-Abe3    4519
FAD-Abe1    4465
FAD-Veh3    4051
WT-Veh2     3442
WT-Veh1     2655
WT-Abe1     2322
WT-Abe3     2166
WT-Abe2     2137
WT-Veh3     1830
Name: count, dtype: int64

In [1]:
#FAD Veh DAM vs HOME
import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

# ============================================================
# PATHS
# ============================================================
ADATA_PATH = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/combined/"
    "sub_adata_microglia_leiden6_noOutside_reclustered_gatex2_with_miranda_"
    "DAMvHOME_damifnhome_combined_withCounts_combinedcleaned_UPDATE_palette2.h5ad"
)

OUT_DIR = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/"
    "final_plots/microglia_subset/allGenes_singleMouseTPC_tables_04032026"
)
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
STATE_COL = "miranda_cell_type_refined_combinedUpdate"
COND_COL = "condition_consol"
COND_MOUSE_COL = "condition"
COUNTS_LAYER = "counts"

TARGET_COND = "FAD-Veh"
STATE_A = "DAM"
STATE_B = "Homeostatic microglia"

PSEUDOCOUNT = 1e-6
PVALUE_FLOOR = 1e-300

# ============================================================
# HELPERS
# ============================================================
def extract_mouse_id(condition_value):
    m = re.search(r"(\d+)$", str(condition_value))
    return m.group(1) if m else np.nan

def get_counts_matrix(ad, layer_name=COUNTS_LAYER):
    if layer_name in ad.layers:
        return ad.layers[layer_name]
    return ad.X

def get_gene_vector(X, idx, gi):
    if sp.issparse(X):
        return X[idx, gi].toarray().ravel()
    return np.asarray(X[idx, gi]).ravel()

def add_bh_fdr(df, p_col="pval"):
    df = df.copy()
    pvals = df[p_col].to_numpy(dtype=float)
    df["p_adj"] = np.nan
    valid = np.isfinite(pvals)

    if valid.sum() > 0:
        _, padj, _, _ = multipletests(pvals[valid], method="fdr_bh")
        df.loc[df.index[valid], "p_adj"] = padj

    df["pval_clipped"] = df["pval"].clip(lower=PVALUE_FLOOR)
    df["neglog10_p"] = -np.log10(df["pval_clipped"])
    df["p_adj_clipped"] = df["p_adj"].clip(lower=PVALUE_FLOOR)
    df["neglog10_p_adj"] = -np.log10(df["p_adj_clipped"])
    return df

# ============================================================
# LOAD / FILTER
# ============================================================
ad = sc.read_h5ad(ADATA_PATH)

for col in [STATE_COL, COND_COL, COND_MOUSE_COL]:
    if col not in ad.obs.columns:
        raise KeyError(f"Missing required obs column: {col}")

if COUNTS_LAYER not in ad.layers:
    raise KeyError(f"adata.layers['{COUNTS_LAYER}'] not found")

ad = ad[
    (ad.obs[COND_COL].astype(str) == TARGET_COND) &
    (ad.obs[STATE_COL].astype(str).isin([STATE_A, STATE_B]))
].copy()

ad.obs["mouse_id"] = ad.obs[COND_MOUSE_COL].map(extract_mouse_id)
ad = ad[ad.obs.dropna(subset=[STATE_COL, COND_MOUSE_COL, "mouse_id"]).index].copy()

print("Filtered adata shape:", ad.shape)
print(ad.obs[STATE_COL].value_counts(dropna=False))
print(ad.obs["mouse_id"].value_counts(dropna=False).sort_index())

# ============================================================
# MAIN
# ============================================================
X = get_counts_matrix(ad, COUNTS_LAYER)
genes = ad.var_names.to_list()

state_vals = ad.obs[STATE_COL].astype(str).values
mouse_vals = ad.obs["mouse_id"].astype(str).values

idxA = np.where(state_vals == STATE_A)[0]
idxB = np.where(state_vals == STATE_B)[0]

mouse_ids_a = sorted(np.unique(mouse_vals[idxA]).tolist(), key=lambda x: int(x))
mouse_ids_b = sorted(np.unique(mouse_vals[idxB]).tolist(), key=lambda x: int(x))

rows = []

for gi, gene in enumerate(genes):
    a_vals = get_gene_vector(X, idxA, gi)
    b_vals = get_gene_vector(X, idxB, gi)

    try:
        stat, p = ttest_ind(a_vals, b_vals, equal_var=False, nan_policy="omit")
    except Exception:
        stat, p = (np.nan, np.nan)

    mean_a = float(np.mean(a_vals))
    mean_b = float(np.mean(b_vals))
    log2fc = np.log2((mean_a + PSEUDOCOUNT) / (mean_b + PSEUDOCOUNT))

    row = {
        "gene": gene,
        "condition": TARGET_COND,
        "state_A": STATE_A,
        "state_B": STATE_B,
        "n_cells_A": int(idxA.size),
        "n_cells_B": int(idxB.size),
        f"mean_{STATE_A}": mean_a,
        f"mean_{STATE_B}": mean_b,
        "log2FC": log2fc,
        "t_stat": stat,
        "pval": p,
        "state_col_used": STATE_COL,
        "condition_col_used": COND_COL,
    }

    for m in mouse_ids_a:
        idxA_m = idxA[mouse_vals[idxA] == str(m)]
        vals_m = get_gene_vector(X, idxA_m, gi) if idxA_m.size > 0 else np.array([])
        row[f"{STATE_A}_mouse{m}_meanTPC"] = float(np.mean(vals_m)) if vals_m.size > 0 else np.nan
        row[f"{STATE_A}_mouse{m}_nCells"] = int(idxA_m.size)

    for m in mouse_ids_b:
        idxB_m = idxB[mouse_vals[idxB] == str(m)]
        vals_m = get_gene_vector(X, idxB_m, gi) if idxB_m.size > 0 else np.array([])
        row[f"{STATE_B}_mouse{m}_meanTPC"] = float(np.mean(vals_m)) if vals_m.size > 0 else np.nan
        row[f"{STATE_B}_mouse{m}_nCells"] = int(idxB_m.size)

    rows.append(row)

res = pd.DataFrame(rows)
res = add_bh_fdr(res, p_col="pval")

out_csv = os.path.join(
    OUT_DIR,
    "allGenes_singleMouseTPC_FAD-Veh_DAM_vs_Homeostatic.csv"
)
res.to_csv(out_csv, index=False)

print("Wrote:", out_csv)

Filtered adata shape: (18151, 347)
miranda_cell_type_refined_combinedUpdate
Homeostatic microglia    10704
DAM                       7447
Name: count, dtype: int64
mouse_id
1    6318
2    7782
3    4051
Name: count, dtype: int64
Wrote: /data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/final_plots/microglia_subset/allGenes_singleMouseTPC_tables_04032026/allGenes_singleMouseTPC_FAD-Veh_DAM_vs_Homeostatic.csv


In [2]:
import os
import re
import glob
import pandas as pd

# ============================================================
# STACK EXISTING allGenes_singleMouseTPC TABLES
# ============================================================
OUT_DIR = (
    "/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/"
    "final_plots/microglia_subset/allGenes_singleMouseTPC_tables_04032026"
)

STACKED_OUT_CSV = os.path.join(
    OUT_DIR,
    "allGenes_singleMouseTPC_COMBINED_stacked.csv"
)

# grab all per-comparison tables, but avoid accidentally re-reading prior combined files
csv_files = sorted(
    [
        f for f in glob.glob(os.path.join(OUT_DIR, "allGenes_singleMouseTPC_*.csv"))
        if "COMBINED" not in os.path.basename(f)
    ]
)

if len(csv_files) == 0:
    raise FileNotFoundError(f"No matching CSVs found in: {OUT_DIR}")

dfs = []

for f in csv_files:
    df = pd.read_csv(f)

    base = os.path.basename(f).replace(".csv", "")

    # preserve original structure; just add a couple of helper columns
    df["source_file"] = os.path.basename(f)

    # if these columns already exist from your newer code, make a clean label from them
    if {"group", "condition_A", "condition_B"}.issubset(df.columns):
        df["comparison_label"] = (
            df["group"].astype(str) + ": " +
            df["condition_A"].astype(str) + " vs " +
            df["condition_B"].astype(str)
        )
    # fallback for older DAM vs HOME style table
    elif {"condition", "state_A", "state_B"}.issubset(df.columns):
        df["comparison_label"] = (
            df["condition"].astype(str) + ": " +
            df["state_A"].astype(str) + " vs " +
            df["state_B"].astype(str)
        )
    else:
        df["comparison_label"] = base

    dfs.append(df)

combined = pd.concat(dfs, axis=0, ignore_index=True, sort=False)

# put identifying columns first, keep everything else in original order after that
priority_cols = [
    "comparison_label",
    "source_file",
    "group",
    "condition_A",
    "condition_B",
    "condition",
    "state_A",
    "state_B",
    "gene",
]
remaining_cols = [c for c in combined.columns if c not in priority_cols]
combined = combined[[c for c in priority_cols if c in combined.columns] + remaining_cols]

combined.to_csv(STACKED_OUT_CSV, index=False)

print("Wrote combined stacked table:")
print(STACKED_OUT_CSV)
print("Shape:", combined.shape)
print("Source tables stacked:", len(csv_files))

Wrote combined stacked table:
/data/projects/oscar/Miranda/results/regatex2/recluster_gatex2/results/final_plots/microglia_subset/allGenes_singleMouseTPC_tables_04032026/allGenes_singleMouseTPC_COMBINED_stacked.csv
Shape: (2776, 64)
Source tables stacked: 8
